# VTK Multi-Label SurfaceNets & Gmsh 3D Conformal Tetrahedral Meshing Pipeline (Graded Size Field Architecture)

Structured, parameterized pipeline taking **Monte-Carlo 3D grain structures** forward from raw multi-label contouring to **watertight 2D conformal surface remeshing** and **fully graded 3D tetrahedral meshing** with six user-configurable boundary-to-center sizing control modes:

### Key Architectural Features & Capabilities:
1. **UPXO Monte-Carlo Potts Simulation Integration (`mcgs` class, Section 2)**:
   - Generates realistic 3D Potts grain structures rather than synthetic Voronoi tessellations.
   - Automated island grain merging and iterative spike voxel removal (`upxo.gsdataops.grid_ops.clean_voxel_spikes_3d`, Section 2 & 2.1).
2. **Deterministic Multi-Label Surface Remeshing (Sections 3 - 12)**:
   - Padded label volume contouring with `vtkDiscreteFlyingEdges3D` / `vtkSurfaceNets3D` and interior Laplacian smoothing.
   - **Technique 1 (Section 7)**: Coupled Seam Smoothing & High-Performance Gmsh 2D Frontal-Delaunay Cap Remeshing with Planar Hole Nesting and `cKDTree` metric node snapping (`tol = 1e-4`).
   - **Technique 2 (Section 12)**: Dynamic Interior Edge Collapse (Technique 2A) + Edge Split (Technique 2B) + Active-Queue 3D Lawson Flipping under strict locked boundary invariance.
   - **Strict 100% Watertightness**: Zero open boundary edges across all 9,319 patches and 1,432 grain shells.
3. **Closed Discrete Surfaces & SurfaceLoop B-Rep Assembly (Section 13)**:
   - Discrete 2D surface registration and signed `SurfaceLoop` assignment per grain volume in Gmsh.
4. **Comprehensive Boundary-to-Center Sizing Field Engine (Section 14)**:
   - **Mode 1 (`constant_auto`)**: Automatically estimates element size from surface triangulation edge length statistics.
   - **Mode 2 (`linear`)**: Linear ramp from fine boundary size $h_{\text{min}}$ to interior core size $h_{\text{max}}$ across $[d_{\text{min}}, d_{\text{max}}]$.
   - **Mode 3 (`exponential`)**: Geometric expansion ratio $r$ per layer away from grain boundaries ($h(d) = h_{\text{min}} \cdot r^{d/d_0}$).
   - **Mode 4 (`sigmoid`)**: $C^1$-continuous cubic Hermite smoothstep S-curve transition.
   - **Mode 5 (`stepped_abs`)**: Stepped distance zones based on physical absolute distance: `{1: [x1, a1], 2: [x2, a2], 3: a3}`.
   - **Mode 6 (`stepped_deq`)**: Grain-adaptive stepped zones based on fraction of grain equivalent spherical diameter $D_{\text{eq}}$: `{1: [f1, a1], 2: [f2, a2], 3: a3}`.
   - **Universal Ceiling (`H_MAX_ALLOWED`)**: Strict global upper threshold capping maximum tetrahedron size across all modes.
5. **3D Conformal Tetrahedral Meshing (Section 14)**:
   - Volume mesh generation via `gmsh.model.mesh.generate(3)` (Delaunay or parallel HXT) with post-meshing Netgen optimization.
6. **3D Tetrahedral Quality Diagnostics & Interactive Crinkle Visualizations (Sections 18 - 20)**:
   - Full SICN, aspect ratio, Jacobian, and positive volume verification ($V > 0$, 0 inverted elements).
   - Interactive 3D PyVista crinkle-clipped cross-sections colored by Grain ID and colored by Element Volume / Sizing Gradient.


## 0. Imports


In [ ]:
from copy import deepcopy
from collections import defaultdict, deque
import os
import time
import math
import numpy as np
import vtk
from vtk.util import numpy_support as ns
import pyvista as pv
import matplotlib.path as mpath
from scipy import sparse
from scipy.spatial import cKDTree
import scipy.ndimage
import cc3d
import gmsh

from upxo.ggrowth.mcgs import mcgs
import upxo.gsdataops.gid_ops as gidOps
from upxo.gsdataops.grid_ops import clean_voxel_spikes_3d


## 1. Control parameters

Tunable parameters grouped by pipeline stage. Configure simulation inputs, surface remeshing thresholds,
and the boundary-to-center 3D sizing control engine here.


In [ ]:
# --- Monte-Carlo simulation / input dashboard --------------------------------
INPUT_DASHBOARD = r'C:\Development\UPXO\upxo_library\src\upxo\demos\gsgen\demo3d1.xls'
TSLICE_INDEX = -1            # temporal slice index (-1 = final Monte-Carlo step)

# --- island grain removal & voxel cleaning ------------------------------------
ISLAND_CONNECTIVITY = 6      # voxel neighborhood connectivity for neighbor graph (6, 18, or 26)
MAX_SPIKE_PASSES = 10        # maximum iterative spike removal passes

# --- padding / background ----------------------------------------------------
BACKGROUND_LABEL = 0
PAD = 1                      # >=1 voxel layer needed for RVE-boundary grains to get capped

# --- interior smoothing (shapes true 3D grain-grain interfaces) -------------
INTERIOR_CONSTRAINT = 1.0    # max per-point movement (voxel units)
INTERIOR_ITERATIONS = 40     # smoothing iterations
INTERIOR_RELAXATION = 0.3    # Laplacian relaxation factor

# --- RVE-face grain boundary (seam) smoothing -------------------------------
SEAM_CONSTRAINT = 0.35       # max per-point movement on RVE boundary faces (voxel units)
SEAM_ITERATIONS = 20         # smoothing iterations along 1D boundary network
SEAM_RELAXATION = 0.30       # Laplacian relaxation factor per pass
CAP_RELAXATION_WEIGHT = 0.30 # relaxation weight for pure cap vertices to maintain triangle quality

# --- RVE-face surface meshing algorithm --------------------------------------
RVE_MESH_ALGORITHM = 'gmsh_frontal_delaunay' # 'gmsh_frontal_delaunay' or 'lawson_delaunay'

# --- numerical tolerances -----------------------------------------------------
COORD_MATCH_DECIMALS = 6     # rounding used to recover point ids after VTK filters
FACE_MATCH_TOL = 1e-6        # tolerance for "is this point exactly on plane P"

# =============================================================================
# --- 3D Tetrahedral Meshing & Boundary-to-Center Sizing Field Control --------
# =============================================================================
# SIZING_MODE options:
#   'constant_auto' : Automatic uniform sizing estimated from surface triangulation edge length
#   'linear'        : Linear ramp from boundary size H_MIN to interior size H_MAX
#   'exponential'   : Geometric growth factor per layer away from grain boundaries
#   'sigmoid'       : C1 smoothstep Hermite S-curve transition
#   'stepped_abs'   : Step zones based on absolute physical distance from grain boundaries
#   'stepped_deq'   : Grain-size-adaptive step zones based on fraction of grain equivalent diameter (Deq)
SIZING_MODE = 'stepped_deq'

# Universal Upper Threshold: maximum allowable tetrahedron edge length across ALL modes
H_MAX_ALLOWED = 3.5

# Parameters for 'linear' and 'sigmoid' modes:
H_MIN = 0.85                 # boundary element size (matches fine surface mesh)
H_MAX = 3.0                  # interior core target element size
D_MIN = 1.0                  # boundary layer plateau thickness (voxel units)
D_MAX = 4.0                  # distance where coarsening reaches full H_MAX (voxel units)

# Parameters for 'exponential' mode:
EXP_GROWTH_RATIO = 1.20      # 20% size growth per layer away from grain boundaries
EXP_D0 = 1.0                 # characteristic layer thickness (voxel units)

# Dictionary for 'stepped_abs' mode (physical distance in voxel units):
# Format: { step_index: [distance_threshold, target_element_size], ..., last_index: rest_of_domain_size }
STEPPED_ABS_DICT = {
    1: [1.5, 0.85],          # 0.0 to 1.5 voxels from GB: h = 0.85
    2: [3.5, 1.80],          # 1.5 to 3.5 voxels from GB: h = 1.80
    3: 3.20                  # > 3.5 voxels: h = 3.20 (capped by H_MAX_ALLOWED)
}

# Dictionary for 'stepped_deq' mode (fraction of grain equivalent diameter Deq):
# Deq = 2 * (3 * V_grain / (4 * pi))^(1/3)
# Format: { step_index: [fraction_of_Deq, target_element_size], ..., last_index: rest_of_grain_interior_size }
STEPPED_DEQ_DICT = {
    1: [0.10, 0.85],         # 0% to 10% of Deq: h = 0.85
    2: [0.25, 1.80],         # 10% to 25% of Deq: h = 1.80
    3: 3.20                  # > 25% of Deq (core): h = 3.20 (capped by H_MAX_ALLOWED)
}

# Gmsh 3D Volume Meshing Options:
GMSH_3D_ALGORITHM = 1        # 1: Delaunay (standard robust), 10: HXT (parallel multi-threaded Delaunay)
GMSH_OPTIMIZE_NETGEN = True  # Run post-meshing 3D Netgen optimizer
ELEMENT_ORDER = 1            # 1: C3D4 (linear 4-node tets), 2: C3D10 (quadratic 10-node tets)


## 2. Monte-Carlo simulation and island grain removal

1. Runs the UPXO Monte-Carlo Potts grain growth simulation (`mcgs.simulate()`).
2. Extracts the labeled feature image (`lgi` / `lfi`) at the chosen temporal slice.
3. **Detects and merges island grains**: An island grain touches no domain boundary face.
   These grains are merged into adjacent parent host grains.
4. Re-indexes the cleaned grain structure into continuous IDs `1..K`.


In [ ]:
def run_mcgs_simulation(dashboard_path, tslice_index=-1):
    """Instantiate mcgs class, run Potts grain growth, and extract labeled feature volume."""
    print(f"Loading Monte-Carlo dashboard: {dashboard_path}")
    pxt = mcgs(study='independent', input_dashboard=dashboard_path, display_messages=False)
    pxt.simulate(verbose=False)
    
    tslice = pxt.m[tslice_index]
    gstslice = pxt.gs[tslice]
    gstslice.char_morphology_of_grains(label_str_order=3, force_compute=True)
    
    lfi = gstslice.lgi.copy()
    return pxt, gstslice, lfi


def remove_island_grains(lfi, connectivity=6):
    """
    Detect and remove internal island grains by merging them into their parent host grain.
    """
    lfi_clean = lfi.copy()
    struct_elem = scipy.ndimage.generate_binary_structure(3, 1) if connectivity == 6 else np.ones((3, 3, 3), dtype=bool)
    iteration = 0
    total_merged = 0
    
    while True:
        iteration += 1
        boundary_fids = set(gidOps.find_boundary_fids3d(lfi_clean).tolist())
        internal_fids = [int(g) for g in np.unique(lfi_clean) if g not in boundary_fids]
        
        if not internal_fids:
            break
            
        sizes = {gid: int((lfi_clean == gid).sum()) for gid in internal_fids}
        internal_fids_sorted = sorted(internal_fids, key=lambda g: sizes[g])
        
        islands_to_merge = {}
        for gid in internal_fids_sorted:
            if not np.any(lfi_clean == gid):
                continue
            gid_mask = (lfi_clean == gid)
            dilated = scipy.ndimage.binary_dilation(gid_mask, structure=struct_elem)
            neighbor_voxels = lfi_clean[dilated & ~gid_mask]
            
            if len(neighbor_voxels) == 0:
                continue
                
            unique_neighs, counts = np.unique(neighbor_voxels, return_counts=True)
            boundary_neighs = [n for n in unique_neighs if n in boundary_fids]
            if boundary_neighs:
                best_parent = max(boundary_neighs, key=lambda n: counts[list(unique_neighs).index(n)])
            else:
                best_parent = unique_neighs[np.argmax(counts)]
                
            islands_to_merge[gid] = int(best_parent)
            lfi_clean[gid_mask] = best_parent
            total_merged += 1
            
        print(f"  Island removal iteration {iteration}: detected & merged {len(islands_to_merge)} island(s)")
        if not islands_to_merge:
            break
            
    unique_ids = np.unique(lfi_clean)
    mapping = {old_id: new_id for new_id, old_id in enumerate(unique_ids, start=1)}
    clean_lfi = np.vectorize(mapping.get)(lfi_clean).astype(np.uint16)
    grain_ids = np.unique(clean_lfi)
    
    return clean_lfi, grain_ids, total_merged


# Execute simulation and island cleaning
pxt, gstslice, raw_lfi = run_mcgs_simulation(INPUT_DASHBOARD, TSLICE_INDEX)
initial_unique = np.unique(raw_lfi)
print(f"Raw simulation: domain {raw_lfi.shape}, {len(initial_unique)} initial grains")

clean_lfi, grain_ids, n_merged = remove_island_grains(raw_lfi, ISLAND_CONNECTIVITY)
print(f"Island removal: {n_merged} island grain(s) merged into parent grains.")
print(f"Final grain structure: {len(grain_ids)} grains, IDs: {grain_ids.min()}..{grain_ids.max()}")


## 2.1 Iterative Spike Voxel Removal (UPXO grid_ops.clean_voxel_spikes_3d)

Applies iterative spike removal to eliminate 0-face-connected voxel artifacts and secondary singularities.


In [ ]:
def remove_spikes_iterative(lfi, max_passes=10, verbose=True):
    """
    Iteratively apply UPXO's clean_voxel_spikes_3d until convergence.
    """
    total_spikes = 0
    lfi_clean = lfi.copy()
    for it in range(1, max_passes + 1):
        lfi_clean, n_spikes = clean_voxel_spikes_3d(lfi_clean)
        total_spikes += n_spikes
        if verbose:
            print(f"  Spike cleaning pass {it}: {n_spikes} spike voxel(s) removed (cumulative: {total_spikes})")
        if n_spikes == 0:
            if verbose:
                print(f"  Iterative spike removal converged after {it} pass(es) -- zero remaining spikes.")
            break
    return lfi_clean, total_spikes, it


print("Applying iterative UPXO clean_voxel_spikes_3d...")
clean_lfi, n_spikes_total, n_passes = remove_spikes_iterative(clean_lfi, max_passes=MAX_SPIKE_PASSES, verbose=True)

# Ensure all grains remain connected to domain boundary
while True:
    boundary_fids = set(gidOps.find_boundary_fids3d(clean_lfi).tolist())
    internal_fids = [int(g) for g in np.unique(clean_lfi) if g not in boundary_fids]
    if not internal_fids:
        break
    clean_lfi, _, extra_merged = remove_island_grains(clean_lfi, ISLAND_CONNECTIVITY)
    print(f"  Post-spike island cleanup: merged {extra_merged} internal grain(s).")

unique_ids = np.unique(clean_lfi)
mapping = {old_id: new_id for new_id, old_id in enumerate(unique_ids, start=1)}
clean_lfi = np.vectorize(mapping.get)(clean_lfi).astype(np.uint16)
grain_ids = np.unique(clean_lfi)
print(f"Final grain structure after iterative spike cleaning: {len(grain_ids)} grains, IDs: {grain_ids.min()}..{grain_ids.max()}")


## 3. Padded label volume -> raw multi-label contour

Generates raw multi-label contour using `vtkSurfaceNets3D` with pre-smoothing quad triangulation.


In [ ]:
def pad_with_background(labels, background_label, pad):
    return np.pad(labels, pad, mode='constant', constant_values=background_label)


def labels_to_image(padded_labels, spacing, pad):
    origin = (-pad * spacing[0], -pad * spacing[1], -pad * spacing[2])
    img = vtk.vtkImageData()
    img.SetDimensions(*padded_labels.shape)
    img.SetSpacing(*spacing)
    img.SetOrigin(*origin)
    flat = padded_labels.ravel(order='F')
    vtk_arr = ns.numpy_to_vtk(flat, deep=True, array_type=vtk.VTK_UNSIGNED_SHORT)
    vtk_arr.SetName('GrainID')
    img.GetPointData().SetScalars(vtk_arr)
    return img


def contour_surfacenets(img, grain_ids, background_label, smoothing=False):
    sn = vtk.vtkSurfaceNets3D()
    sn.SetInputData(img)
    sn.SetBackgroundLabel(background_label)
    sn.SetNumberOfLabels(len(grain_ids))
    for i, gid in enumerate(grain_ids):
        sn.SetLabel(i, float(gid))
    sn.SetSmoothing(smoothing)
    sn.Update()
    out = vtk.vtkPolyData()
    out.DeepCopy(sn.GetOutput())
    return out


spacing = (1.0, 1.0, 1.0)
padded = pad_with_background(clean_lfi, BACKGROUND_LABEL, PAD)
img = labels_to_image(padded, spacing, PAD)
raw_quad = contour_surfacenets(img, grain_ids, BACKGROUND_LABEL, smoothing=False)

tf_raw = vtk.vtkTriangleFilter()
tf_raw.SetInputData(raw_quad)
tf_raw.PassLinesOff()
tf_raw.PassVertsOff()
tf_raw.Update()
raw = tf_raw.GetOutput()

bl = ns.vtk_to_numpy(raw.GetCellData().GetArray('BoundaryLabels'))
n_bg = sum(1 for pair in np.unique(bl, axis=0) if BACKGROUND_LABEL in pair)
n_gg = sum(1 for pair in np.unique(bl, axis=0) if BACKGROUND_LABEL not in pair)
print(f"raw contour: {raw.GetNumberOfPoints()} points, {raw.GetNumberOfCells()} faces")
print(f"  grain-vs-background interface groups: {n_bg}")
print(f"  grain-vs-grain interface groups:       {n_gg}")


## 4. RVE-plane tagging


In [ ]:
def tag_rve_boundary_axes(mesh, tol):
    bounds = mesh.GetBounds()
    pts = ns.vtk_to_numpy(mesh.GetPoints().GetData())
    axis_clamp = np.full((pts.shape[0], 3), np.nan)
    for a in range(3):
        lo, hi = bounds[2 * a], bounds[2 * a + 1]
        axis_clamp[np.isclose(pts[:, a], lo, atol=tol), a] = lo
        axis_clamp[np.isclose(pts[:, a], hi, atol=tol), a] = hi
    on_boundary = ~np.isnan(axis_clamp).all(axis=1)
    return on_boundary, axis_clamp


raw_pts = ns.vtk_to_numpy(raw.GetPoints().GetData())
xmin, xmax, ymin, ymax, zmin, zmax = raw.GetBounds()
on_boundary, axis_clamp = tag_rve_boundary_axes(raw, FACE_MATCH_TOL)
n_tagged_axes = (~np.isnan(axis_clamp)).sum(axis=1)
n_face = int((n_tagged_axes == 1).sum())
n_edge = int((n_tagged_axes == 2).sum())
n_corner = int((n_tagged_axes == 3).sum())
print(f"{on_boundary.sum()} / {raw.GetNumberOfPoints()} points on an RVE boundary "
      f"({n_face} face-interior, {n_edge} edge, {n_corner} corner)")


## 5. Interior smoothing


In [ ]:
def constrained_smooth_then_project(mesh, axis_clamp, constraint, n_iterations, relaxation):
    m = vtk.vtkPolyData()
    m.DeepCopy(mesh)
    constraints = np.full(m.GetNumberOfPoints(), constraint, dtype=np.float64)
    carr = ns.numpy_to_vtk(constraints, deep=True, array_type=vtk.VTK_DOUBLE)
    carr.SetName('SmoothingConstraints')
    m.GetPointData().AddArray(carr)

    smoother = vtk.vtkConstrainedSmoothingFilter()
    smoother.SetInputData(m)
    smoother.SetConstraintStrategyToConstraintArray()
    smoother.SetInputArrayToProcess(0, 0, 0, vtk.vtkDataObject.FIELD_ASSOCIATION_POINTS,
                                     'SmoothingConstraints')
    smoother.SetNumberOfIterations(n_iterations)
    smoother.SetRelaxationFactor(relaxation)
    smoother.Update()
    out = vtk.vtkPolyData()
    out.DeepCopy(smoother.GetOutput())

    pts = ns.vtk_to_numpy(out.GetPoints().GetData()).copy()
    for a in range(3):
        mask = ~np.isnan(axis_clamp[:, a])
        pts[mask, a] = axis_clamp[mask, a]
    out.GetPoints().SetData(ns.numpy_to_vtk(pts, deep=True, array_type=vtk.VTK_DOUBLE))
    return out


flat_smoothed = constrained_smooth_then_project(
    raw, axis_clamp, INTERIOR_CONSTRAINT, INTERIOR_ITERATIONS, INTERIOR_RELAXATION)

interior_disp = np.linalg.norm(
    ns.vtk_to_numpy(flat_smoothed.GetPoints().GetData())[~on_boundary]
    - raw_pts[~on_boundary], axis=1)
print(f"interior displacement: mean {interior_disp.mean():.3f}, max {interior_disp.max():.3f}")


## 6. Seam extraction


In [ ]:
coord_to_id = {tuple(np.round(p, COORD_MATCH_DECIMALS)): i for i, p in enumerate(raw_pts)}

n_cells = raw.GetNumberOfCells()
cell_point_ids = np.array([
    [raw.GetCell(cid).GetPointId(k) for k in range(raw.GetCell(cid).GetNumberOfPoints())]
    for cid in range(n_cells)
])
bl = ns.vtk_to_numpy(raw.GetCellData().GetArray('BoundaryLabels'))
cell_grain = np.where(bl[:, 0] != BACKGROUND_LABEL, bl[:, 0], bl[:, 1])

face_axis_value = [(0, xmin), (0, xmax), (1, ymin), (1, ymax), (2, zmin), (2, zmax)]
face_names = ['xmin', 'xmax', 'ymin', 'ymax', 'zmin', 'zmax']

seam_edges = set()
seam_point_ids = set()
cap_cell_ids = set()
loop_points_by_face_grain = {}
n_loops_extracted = 0

for (axis_i, value), fname in zip(face_axis_value, face_names):
    on_face_point = np.isclose(raw_pts[:, axis_i], value, atol=FACE_MATCH_TOL)
    is_cap_cell = on_face_point[cell_point_ids].all(axis=1)
    for g in sorted(set(cell_grain[is_cap_cell].tolist())):
        cell_ids = np.where(is_cap_cell & (cell_grain == g))[0]
        cap_cell_ids.update(cell_ids.tolist())

        cells_arr = vtk.vtkCellArray()
        for cid in cell_ids:
            cells_arr.InsertNextCell(raw.GetCell(int(cid)))
        subset = vtk.vtkPolyData()
        subset.SetPoints(raw.GetPoints())
        subset.SetPolys(cells_arr)

        fe = vtk.vtkFeatureEdges()
        fe.SetInputData(subset)
        fe.BoundaryEdgesOn()
        fe.FeatureEdgesOff()
        fe.NonManifoldEdgesOff()
        fe.ManifoldEdgesOff()
        fe.Update()
        seam = fe.GetOutput()
        if seam.GetNumberOfPoints() == 0:
            continue
        n_loops_extracted += 1

        seam_local_pts = ns.vtk_to_numpy(seam.GetPoints().GetData())
        local_to_global = np.array(
            [coord_to_id[tuple(np.round(p, COORD_MATCH_DECIMALS))] for p in seam_local_pts])

        this_loop_pts = set()
        lines = seam.GetLines()
        lines.InitTraversal()
        idlist = vtk.vtkIdList()
        while lines.GetNextCell(idlist):
            p0, p1 = int(local_to_global[idlist.GetId(0)]), int(local_to_global[idlist.GetId(1)])
            seam_edges.add((min(p0, p1), max(p0, p1)))
            seam_point_ids.add(p0)
            seam_point_ids.add(p1)
            this_loop_pts.add(p0)
            this_loop_pts.add(p1)
        loop_points_by_face_grain[(fname, g)] = this_loop_pts

print(f"{n_loops_extracted} (face, grain) seam loops extracted")
print(f"cap cells to discard: {len(cap_cell_ids)} / {n_cells}")
print(f"seam graph: {len(seam_point_ids)} points, {len(seam_edges)} edges")


## 7. High-Performance Coupled Seam Smoothing & Gmsh 2D Frontal-Delaunay Remeshing

Performs coupled seam + cap Laplacian smoothing, inlined 3D Delaunay quad splitting on interior interfaces,
and Gmsh 2D Frontal-Delaunay surface remeshing with planar hole nesting.


In [ ]:
_t_sec7_start = time.perf_counter()

# 1. Fast cell extraction from VTK poly connectivity
_t_extract_start = time.perf_counter()
polys_vtk = flat_smoothed.GetPolys()
conn_list = ns.vtk_to_numpy(polys_vtk.GetData()).tolist()
N_conn = len(conn_list)

cell_pids = []
all_sample_tris = []
idx = 0
while idx < N_conn:
    npts = conn_list[idx]
    pids = conn_list[idx+1 : idx+1+npts]
    cell_pids.append(pids)
    all_sample_tris.append([pids[0], pids[1], pids[2]])
    if npts == 4:
        all_sample_tris.append([pids[0], pids[2], pids[3]])
    idx += npts + 1

all_sample_tris = np.array(all_sample_tris, dtype=np.int64)

seam_ids = np.array(sorted(seam_point_ids), dtype=np.int64)
n_seam = len(seam_ids)
seam_id_to_idx = {pid: i for i, pid in enumerate(seam_ids)}
is_seam = np.zeros(len(raw_pts), dtype=bool)
is_seam[seam_ids] = True

v_to_cells = defaultdict(list)
for cid, tri in enumerate(all_sample_tris):
    if is_seam[tri[0]]: v_to_cells[tri[0]].append(cid)
    if is_seam[tri[1]]: v_to_cells[tri[1]].append(cid)
    if is_seam[tri[2]]: v_to_cells[tri[2]].append(cid)

rows = []
cols = []
vals = []
for p0, p1 in seam_edges:
    i0 = seam_id_to_idx[p0]
    i1 = seam_id_to_idx[p1]
    rows.extend([i0, i1])
    cols.extend([i1, i0])
    vals.extend([1.0, 1.0])

W_seam = sparse.csr_matrix((vals, (rows, cols)), shape=(n_seam, n_seam))
deg = np.array(W_seam.sum(axis=1)).ravel()
deg[deg == 0] = 1.0
inv_deg = sparse.diags(1.0 / deg)
W_seam_norm = inv_deg.dot(W_seam)

cap_tris = np.array([cell_pids[cid][:3] for cid in cap_cell_ids], dtype=np.int64)
cap_adj = defaultdict(set)
for t in cap_tris:
    cap_adj[t[0]].update([t[1], t[2]])
    cap_adj[t[1]].update([t[0], t[2]])
    cap_adj[t[2]].update([t[0], t[1]])

pure_cap_ids = np.array([p for p in np.where(on_boundary)[0] if not is_seam[p]], dtype=np.int64)
n_cap = len(pure_cap_ids)
flat_pts = ns.vtk_to_numpy(flat_smoothed.GetPoints().GetData()).copy()
curr_pts = flat_pts.copy()

rows_c = []
cols_c = []
vals_c = []
for i, p in enumerate(pure_cap_ids):
    neighs = cap_adj[p]
    for q in neighs:
        rows_c.append(i)
        cols_c.append(q)
        vals_c.append(1.0 / len(neighs))

W_cap = sparse.csr_matrix((vals_c, (rows_c, cols_c)), shape=(n_cap, len(flat_pts)))

def fast_min_incident_area(pts, v):
    cids = v_to_cells[v]
    tris = all_sample_tris[cids]
    p0 = pts[tris[:, 0]]
    p1 = pts[tris[:, 1]]
    p2 = pts[tris[:, 2]]
    v1 = p1 - p0
    v2 = p2 - p0
    cx = v1[:, 1] * v2[:, 2] - v1[:, 2] * v2[:, 1]
    cy = v1[:, 2] * v2[:, 0] - v1[:, 0] * v2[:, 2]
    cz = v1[:, 0] * v2[:, 1] - v1[:, 1] * v2[:, 0]
    areas = 0.5 * np.sqrt(cx*cx + cy*cy + cz*cz)
    return np.min(areas)

raw_seam_pts = raw_pts[seam_ids]
axis_clamp_seam = axis_clamp[seam_ids]
clamp_masks = [~np.isnan(axis_clamp_seam[:, a]) for a in range(3)]
clamp_vals = [axis_clamp_seam[clamp_masks[a], a] for a in range(3)]

raw_cap_pts = raw_pts[pure_cap_ids]
axis_clamp_cap = axis_clamp[pure_cap_ids]
clamp_masks_c = [~np.isnan(axis_clamp_cap[:, a]) for a in range(3)]
clamp_vals_c = [axis_clamp_cap[clamp_masks_c[a], a] for a in range(3)]

print(f"Performing coupled seam + cap smoothing (budget {SEAM_CONSTRAINT}, {SEAM_ITERATIONS} iterations)...")
for it in range(SEAM_ITERATIONS):
    neigh_mean = W_seam_norm.dot(curr_pts[seam_ids])
    targets = (1.0 - SEAM_RELAXATION) * curr_pts[seam_ids] + SEAM_RELAXATION * neigh_mean
    for a in range(3):
        targets[clamp_masks[a], a] = clamp_vals[a]
    disp_vecs = targets - raw_seam_pts
    dists = np.linalg.norm(disp_vecs, axis=1)
    over_budget = dists > SEAM_CONSTRAINT
    if np.any(over_budget):
        scale = (SEAM_CONSTRAINT / dists[over_budget])[:, None]
        targets[over_budget] = raw_seam_pts[over_budget] + disp_vecs[over_budget] * scale
    
    step_disp = np.linalg.norm(targets - curr_pts[seam_ids], axis=1)
    curr_pts[seam_ids] = targets
    
    check_mask = step_disp > 0.05
    for idx_check in np.where(check_mask)[0]:
        p = seam_ids[idx_check]
        if fast_min_incident_area(curr_pts, p) < 0.01:
            target = targets[idx_check]
            alpha = 0.5
            while alpha > 0.05:
                curr_pts[p] = (1.0 - alpha) * raw_seam_pts[idx_check] + alpha * target
                if fast_min_incident_area(curr_pts, p) >= 0.01:
                    break
                alpha *= 0.5
            if alpha <= 0.05:
                curr_pts[p] = raw_seam_pts[idx_check]
                
    if it % 2 == 0:
        neigh_mean_c = W_cap.dot(curr_pts)
        targets_c = (1.0 - CAP_RELAXATION_WEIGHT) * curr_pts[pure_cap_ids] + CAP_RELAXATION_WEIGHT * neigh_mean_c
        for a in range(3):
            targets_c[clamp_masks_c[a], a] = clamp_vals_c[a]
        disp_vecs_c = targets_c - raw_cap_pts
        dists_c = np.linalg.norm(disp_vecs_c, axis=1)
        over_budget_c = dists_c > SEAM_CONSTRAINT
        if np.any(over_budget_c):
            scale_c = (SEAM_CONSTRAINT / dists_c[over_budget_c])[:, None]
            targets_c[over_budget_c] = raw_cap_pts[over_budget_c] + disp_vecs_c[over_budget_c] * scale_c
        curr_pts[pure_cap_ids] = targets_c

seam_disp = np.linalg.norm(curr_pts[seam_ids] - raw_pts[seam_ids], axis=1)
seam_plane_residual = 0.0
for a in range(3):
    mask = ~np.isnan(axis_clamp[:, a])
    mask[~np.isin(np.arange(len(raw_pts)), seam_ids)] = False
    seam_plane_residual = max(seam_plane_residual, np.abs(curr_pts[mask, a] - axis_clamp[mask, a]).max())
print(f"seam plane residual: {seam_plane_residual:.1e}  (must be 0)")
print(f"seam displacement: mean {seam_disp.mean():.3f}, max {seam_disp.max():.3f}")

# 3. Fast 3D Delaunay Quad Splitting on Interior Grain-Grain Interfaces
raw_bl = ns.vtk_to_numpy(flat_smoothed.GetCellData().GetArray('BoundaryLabels'))
is_interior_tri = (raw_bl[:, 0] != BACKGROUND_LABEL) & (raw_bl[:, 1] != BACKGROUND_LABEL)

all_mesh_vertices = curr_pts.tolist()
all_mesh_triangles = []
all_mesh_bl = []

def fast_triangle_min_angle_deg(p0, p1, p2):
    v0 = p1 - p0
    v1 = p2 - p1
    v2 = p0 - p2
    l0 = np.sqrt(v0[0]*v0[0] + v0[1]*v0[1] + v0[2]*v0[2])
    l1 = np.sqrt(v1[0]*v1[0] + v1[1]*v1[1] + v1[2]*v1[2])
    l2 = np.sqrt(v2[0]*v2[0] + v2[1]*v2[1] + v2[2]*v2[2])
    if l0 < 1e-12 or l1 < 1e-12 or l2 < 1e-12:
        return 0.0
    dot0 = -(v2[0]*v0[0] + v2[1]*v0[1] + v2[2]*v0[2])
    dot1 = -(v0[0]*v1[0] + v0[1]*v1[1] + v0[2]*v1[2])
    dot2 = -(v1[0]*v2[0] + v1[1]*v2[1] + v1[2]*v2[2])
    cos0 = min(1.0, max(-1.0, dot0 / (l2 * l0)))
    cos1 = min(1.0, max(-1.0, dot1 / (l0 * l1)))
    cos2 = min(1.0, max(-1.0, dot2 / (l1 * l2)))
    return np.degrees(np.arccos(max(cos0, cos1, cos2)))

interior_cids = np.where(is_interior_tri)[0]
for cid in interior_cids:
    pids = cell_pids[cid]
    lbl = raw_bl[cid]
    if len(pids) == 4:
        p0, p1, p2, p3 = pids
        pt0, pt1, pt2, pt3 = curr_pts[p0], curr_pts[p1], curr_pts[p2], curr_pts[p3]
        qA = min(fast_triangle_min_angle_deg(pt0, pt1, pt2), fast_triangle_min_angle_deg(pt0, pt2, pt3))
        qB = min(fast_triangle_min_angle_deg(pt0, pt1, pt3), fast_triangle_min_angle_deg(pt1, pt2, pt3))
        if qB > qA:
            all_mesh_triangles.append([p0, p1, p3])
            all_mesh_bl.append(lbl)
            all_mesh_triangles.append([p1, p2, p3])
            all_mesh_bl.append(lbl)
        else:
            all_mesh_triangles.append([p0, p1, p2])
            all_mesh_bl.append(lbl)
            all_mesh_triangles.append([p0, p2, p3])
            all_mesh_bl.append(lbl)
    else:
        all_mesh_triangles.append([pids[0], pids[1], pids[2]])
        all_mesh_bl.append(lbl)

# 4. Eulerian Cycle Extraction, AABB Hole Nesting & Gmsh 2D Frontal-Delaunay Remeshing
def extract_simple_cycles_eulerian(boundary_edges):
    adj = defaultdict(list)
    for u, v in boundary_edges:
        adj[u].append(v)
        adj[v].append(u)
    edge_count = defaultdict(int)
    for u, v in boundary_edges:
        edge_count[(min(u, v), max(u, v))] += 1
    cycles = []
    for u0, v0 in boundary_edges:
        e0 = (min(u0, v0), max(u0, v0))
        if edge_count[e0] == 0:
            continue
        circuit = []
        stack = [u0]
        while stack:
            curr = stack[-1]
            found = False
            while adj[curr]:
                neigh = adj[curr].pop()
                e = (min(curr, neigh), max(curr, neigh))
                if edge_count[e] > 0:
                    edge_count[e] -= 1
                    stack.append(neigh)
                    found = True
                    break
            if not found:
                circuit.append(stack.pop())
        pos = {}
        sub = []
        for v in circuit:
            if v in pos:
                idx = pos[v]
                loop = sub[idx:]
                for x in loop:
                    pos.pop(x, None)
                sub = sub[:idx]
                if len(loop) >= 3:
                    cycles.append(loop)
            pos[v] = len(sub)
            sub.append(v)
    return cycles

def get_loop_2d(loop_pids, pts_3d, axis_i):
    axes_2d = [a for a in [0, 1, 2] if a != axis_i]
    return pts_3d[loop_pids][:, axes_2d]

def get_signed_area_2d(pts_2d):
    x, y = pts_2d[:, 0], pts_2d[:, 1]
    return 0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)

def classify_loop_for_grain(loop_pids, curr_pts, axis_i, grain_id, face_cell_centers_2d, face_cell_grains):
    axes_2d = [a for a in [0, 1, 2] if a != axis_i]
    loop_pts_2d = curr_pts[loop_pids][:, axes_2d]
    min_x, min_y = np.min(loop_pts_2d, axis=0)
    max_x, max_y = np.max(loop_pts_2d, axis=0)
    in_box = (
        (face_cell_centers_2d[:, 0] >= min_x) & (face_cell_centers_2d[:, 0] <= max_x) &
        (face_cell_centers_2d[:, 1] >= min_y) & (face_cell_centers_2d[:, 1] <= max_y)
    )
    if not np.any(in_box):
        return 'OUTER'
    sub_centers = face_cell_centers_2d[in_box]
    sub_grains = face_cell_grains[in_box]
    path = mpath.Path(loop_pts_2d)
    inside_mask = path.contains_points(sub_centers)
    if not np.any(inside_mask):
        return 'OUTER'
    grains_inside = sub_grains[inside_mask]
    same_grain_ratio = np.mean(grains_inside == grain_id)
    return 'HOLE' if same_grain_ratio < 0.5 else 'OUTER'

def create_grain_surfaces_with_holes(loops, curr_pts, axis_i, grain_id, face_cell_centers_2d, face_cell_grains, get_or_add_point, get_or_add_line):
    axes_2d = [a for a in [0, 1, 2] if a != axis_i]
    if not loops:
        return []
    if len(loops) == 1:
        loop = list(loops[0])
        area = get_signed_area_2d(get_loop_2d(loop, curr_pts, axis_i))
        if area < 0:
            loop = loop[::-1]
        curve_tags = []
        L = len(loop)
        for k in range(L):
            tag1 = get_or_add_point(curr_pts[loop[k]])
            tag2 = get_or_add_point(curr_pts[loop[(k + 1) % L]])
            curve_tags.append(get_or_add_line(tag1, tag2))
        try:
            cloop = gmsh.model.geo.addCurveLoop(curve_tags)
            surf = gmsh.model.geo.addPlaneSurface([cloop])
            return [surf]
        except Exception as e:
            return []
            
    loop_data = []
    for l_idx, loop in enumerate(loops):
        pts_2d = curr_pts[loop][:, axes_2d]
        x, y = pts_2d[:, 0], pts_2d[:, 1]
        area = 0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)
        l_type = classify_loop_for_grain(loop, curr_pts, axis_i, grain_id, face_cell_centers_2d, face_cell_grains)
        loop_data.append({'loop': loop, 'pts_2d': pts_2d, 'area': area, 'abs_area': abs(area), 'path': mpath.Path(pts_2d), 'type': l_type})
        
    for i in range(len(loop_data)):
        for j in range(i + 1, len(loop_data)):
            if len(set(loop_data[i]['loop']).intersection(set(loop_data[j]['loop']))) > 0:
                loop_data[i]['type'] = 'OUTER'
                loop_data[j]['type'] = 'OUTER'

    outer_loops = [d for d in loop_data if d['type'] == 'OUTER']
    hole_loops = [d for d in loop_data if d['type'] == 'HOLE']
    if not outer_loops:
        loop_data.sort(key=lambda d: d['abs_area'], reverse=True)
        loop_data[0]['type'] = 'OUTER'
        outer_loops = [loop_data[0]]
        hole_loops = loop_data[1:]
        
    outer_loops.sort(key=lambda d: d['abs_area'], reverse=True)
    surfaces = []
    assigned_holes = [False] * len(hole_loops)
    for outer in outer_loops:
        contained_holes = []
        for h_idx, h in enumerate(hole_loops):
            if not assigned_holes[h_idx]:
                centroid = np.mean(h['pts_2d'], axis=0)
                is_inside = outer['path'].contains_point(centroid) or (np.mean(outer['path'].contains_points(h['pts_2d'])) > 0.5)
                if is_inside:
                    contained_holes.append(h)
                    assigned_holes[h_idx] = True
        surfaces.append((outer, contained_holes))
        
    for h_idx, h in enumerate(hole_loops):
        if not assigned_holes[h_idx]:
            surfaces[0][1].append(h)
            
    surf_tags = []
    for outer, holes in surfaces:
        o_loop = list(outer['loop'])
        if outer['area'] < 0:
            o_loop = o_loop[::-1]
        curve_tags = []
        L = len(o_loop)
        for k in range(L):
            tag1 = get_or_add_point(curr_pts[o_loop[k]])
            tag2 = get_or_add_point(curr_pts[o_loop[(k + 1) % L]])
            curve_tags.append(get_or_add_line(tag1, tag2))
        cloop_list = []
        try:
            o_cloop = gmsh.model.geo.addCurveLoop(curve_tags)
            cloop_list.append(o_cloop)
        except Exception:
            continue
        for h in holes:
            h_loop = list(h['loop'])
            if h['area'] > 0:
                h_loop = h_loop[::-1]
            h_curve_tags = []
            Lh = len(h_loop)
            for k in range(Lh):
                tag1 = get_or_add_point(curr_pts[h_loop[k]])
                tag2 = get_or_add_point(curr_pts[h_loop[(k + 1) % Lh]])
                h_curve_tags.append(get_or_add_line(tag1, tag2))
            try:
                h_cloop = gmsh.model.geo.addCurveLoop(h_curve_tags)
                cloop_list.append(h_cloop)
            except Exception:
                pass
        try:
            s_tag = gmsh.model.geo.addPlaneSurface(cloop_list)
            surf_tags.append(s_tag)
        except Exception:
            pass
    return surf_tags

non_interior_cids = np.where(~is_interior_tri)[0]
if RVE_MESH_ALGORITHM == 'gmsh_frontal_delaunay':
    print("Generating 2D Frontal-Delaunay meshes with hole nesting for all 6 RVE boundary faces...")
    face_axis_dict = dict(zip(face_names, face_axis_value))
    for fname, (axis_i, face_val) in face_axis_dict.items():
        on_face = np.isclose(curr_pts[:, axis_i], face_val, atol=FACE_MATCH_TOL)
        face_cap_cids = [cid for cid in non_interior_cids if on_face[cell_pids[cid]].all()]
        face_grain = [int(raw_bl[cid, 0] if raw_bl[cid, 0] != BACKGROUND_LABEL else raw_bl[cid, 1]) for cid in face_cap_cids]
        unique_grains_face = sorted(set(face_grain))
        axes_2d = [a for a in [0, 1, 2] if a != axis_i]
        face_cell_centers_2d = np.array([curr_pts[cell_pids[cid]].mean(axis=0)[axes_2d] for cid in face_cap_cids])
        face_cell_grains = np.array(face_grain)
        
        gmsh.initialize()
        gmsh.option.setNumber('General.Terminal', 0)
        gmsh.option.setNumber('General.Verbosity', 1)
        gmsh.option.setNumber('Mesh.Algorithm', 6)
        gmsh.option.setNumber('Mesh.MeshSizeFactor', 1.0)
        gmsh.model.add(f"face_{fname}")
        
        pt_coord_to_tag = {}
        def get_or_add_point(p):
            key = tuple(np.round(p, 5))
            if key in pt_coord_to_tag:
                return pt_coord_to_tag[key]
            tag = gmsh.model.geo.addPoint(p[0], p[1], p[2], 1.0)
            pt_coord_to_tag[key] = tag
            return tag

        line_key_to_tag = {}
        def get_or_add_line(tag1, tag2):
            t_min, t_max = min(tag1, tag2), max(tag1, tag2)
            key = (t_min, t_max)
            if key not in line_key_to_tag:
                ltag = gmsh.model.geo.addLine(t_min, t_max)
                gmsh.model.geo.mesh.setTransfiniteCurve(ltag, 2)
                line_key_to_tag[key] = ltag
            base_tag = line_key_to_tag[key]
            return base_tag if tag1 == t_min else -base_tag

        grain_surface_tags = {}
        cids_by_grain = defaultdict(list)
        for i, gr in enumerate(face_grain):
            cids_by_grain[gr].append(face_cap_cids[i])
            
        for g in unique_grains_face:
            cids = cids_by_grain[g]
            edge_cnt = defaultdict(int)
            for cid in cids:
                pids = cell_pids[cid]
                if len(pids) == 4:
                    for u, v in [(pids[0], pids[1]), (pids[1], pids[2]), (pids[2], pids[3]), (pids[3], pids[0])]:
                        edge_cnt[(min(u, v), max(u, v))] += 1
                else:
                    for i in range(3):
                        edge_cnt[(min(pids[i], pids[(i+1)%3]), max(pids[i], pids[(i+1)%3]))] += 1
            b_edges = [e for e, cnt in edge_cnt.items() if cnt == 1]
            loops = extract_simple_cycles_eulerian(b_edges)
            surf_tags = create_grain_surfaces_with_holes(
                loops, curr_pts, axis_i, g, face_cell_centers_2d, face_cell_grains, get_or_add_point, get_or_add_line
            )
            grain_surface_tags[g] = surf_tags
            
        gmsh.model.geo.synchronize()
        gmsh.model.mesh.generate(2)
        node_tags, coords, _ = gmsh.model.mesh.getNodes()
        nodes_remeshed = np.array(coords).reshape(-1, 3)
        nodes_remeshed[:, axis_i] = face_val
        
        SNAP_TOL = 1e-4
        spatial_tree = cKDTree(all_mesh_vertices)
        dists, nearest_idxs = spatial_tree.query(nodes_remeshed, distance_upper_bound=SNAP_TOL)
        loc_tag_to_global_vidx = {}
        for tag, p, d, nearest in zip(node_tags, nodes_remeshed, dists, nearest_idxs):
            if d <= SNAP_TOL:
                loc_tag_to_global_vidx[tag] = int(nearest)
            else:
                new_vidx = len(all_mesh_vertices)
                all_mesh_vertices.append(p.tolist())
                loc_tag_to_global_vidx[tag] = new_vidx
                
        n_face_tris = 0
        for g, surfs in grain_surface_tags.items():
            lbl = [float(g), 0.0]
            for s in surfs:
                elem_types, elem_tags, elem_node_tags = gmsh.model.mesh.getElements(2, s)
                if len(elem_node_tags) > 0:
                    raw_tags = np.array(elem_node_tags[0]).reshape(-1, 3)
                    for tri_tags in raw_tags:
                        tri_vidxs = [loc_tag_to_global_vidx[tag] for tag in tri_tags]
                        all_mesh_triangles.append(tri_vidxs)
                        all_mesh_bl.append(lbl)
                        n_face_tris += 1
        print(f"  Face {fname}: {n_face_tris} Frontal-Delaunay triangles generated")
        gmsh.finalize()

all_mesh_vertices = np.array(all_mesh_vertices, dtype=np.float64)
all_mesh_triangles = np.array(all_mesh_triangles, dtype=np.int64)
all_mesh_bl = np.array(all_mesh_bl, dtype=np.float64)

stitched = vtk.vtkPolyData()
pts_vtk = ns.numpy_to_vtk(all_mesh_vertices, deep=True, array_type=vtk.VTK_DOUBLE)
points = vtk.vtkPoints()
points.SetData(pts_vtk)
stitched.SetPoints(points)

n_tri = len(all_mesh_triangles)
cell_conn = np.empty((n_tri, 4), dtype=np.int64)
cell_conn[:, 0] = 3
cell_conn[:, 1:] = all_mesh_triangles
ca = vtk.vtkCellArray()
ca.SetCells(n_tri, ns.numpy_to_vtkIdTypeArray(cell_conn.ravel(), deep=True))
stitched.SetPolys(ca)

bl_arr = ns.numpy_to_vtk(all_mesh_bl, deep=True, array_type=vtk.VTK_DOUBLE)
bl_arr.SetName('BoundaryLabels')
stitched.GetCellData().AddArray(bl_arr)
stitched.GetCellData().SetScalars(bl_arr)

g1 = all_mesh_bl[:, 0]
g2 = all_mesh_bl[:, 1]
final_cell_grain = np.where(g1 != BACKGROUND_LABEL, g1, g2)
grain_cell_arr = ns.numpy_to_vtk(final_cell_grain, deep=True)
grain_cell_arr.SetName('grain')
stitched.GetCellData().AddArray(grain_cell_arr)

final_tris = all_mesh_triangles
final_bl = all_mesh_bl
print(f"stitched mesh: {stitched.GetNumberOfPoints()} points, {stitched.GetNumberOfCells()} cells")


## 8. Quality Diagnostics Dashboard


In [ ]:
def count_open_edges(mesh):
    fe = vtk.vtkFeatureEdges()
    fe.SetInputData(mesh)
    fe.BoundaryEdgesOn()
    fe.FeatureEdgesOff()
    fe.NonManifoldEdgesOff()
    fe.ManifoldEdgesOff()
    fe.Update()
    return fe.GetOutput().GetNumberOfLines()

n_open_after_fill = count_open_edges(stitched)
print("=== mesh sizes ===")
print(f"  raw contour:        {raw.GetNumberOfPoints():6d} points,  {raw.GetNumberOfCells():6d} cells")
print(f"  stitched (final):   {stitched.GetNumberOfPoints():6d} points,  {stitched.GetNumberOfCells():6d} cells")
print(f"=== watertightness ===")
print(f"  open boundary edges:          {n_open_after_fill}  (target: 0)")
watertight = (n_open_after_fill == 0)
print(f"  status: {'WATERTIGHT' if watertight else f'NOT WATERTIGHT -- {n_open_after_fill} open edges'}")
assert watertight, f"Mesh not watertight: {n_open_after_fill} open edges remain"


## 9. Visualize: crinkle-clipped stitched result


In [ ]:
stitched_grid = pv.wrap(stitched)
stitched_grid.cell_data['grain'] = ns.vtk_to_numpy(stitched.GetCellData().GetArray('grain')).astype(float)

pvp = pv.Plotter(window_size=(800, 700))
pvp.set_background('white')
pvp.add_mesh_clip_plane(stitched_grid, normal='x', crinkle=True, scalars='grain', cmap='tab20', show_edges=True)
pvp.add_title(f'Monte-Carlo Stitched RVE ({len(grain_ids)} grains) -- crinkle clip along x', font_size=10)
pvp.add_axes()
pvp.show()


## 10. Grain Boundary Surface Patch Segmentation


In [ ]:
bounds = raw.GetBounds()
xmin, xmax, ymin, ymax, zmin, zmax = bounds
face_axis_value = [(0, xmin), (0, xmax), (1, ymin), (1, ymax), (2, zmin), (2, zmax)]
face_names = ['xmin', 'xmax', 'ymin', 'ymax', 'zmin', 'zmax']

final_points = ns.vtk_to_numpy(stitched.GetPoints().GetData())
n_cells = stitched.GetNumberOfCells()
bl = ns.vtk_to_numpy(stitched.GetCellData().GetArray('BoundaryLabels'))

interior_triangles = []
interior_patch_names = []
interior_grain_pairs = []
cap_triangles = []
cap_patch_names = []
cap_grain_tags = []

int_pair_to_tris = defaultdict(list)
for cid in range(n_cells):
    cell = stitched.GetCell(cid)
    ids = [cell.GetPointId(0), cell.GetPointId(1), cell.GetPointId(2)]
    g1, g2 = int(bl[cid, 0]), int(bl[cid, 1])
    if g1 != 0 and g2 != 0:
        pair = (min(g1, g2), max(g1, g2))
        int_pair_to_tris[pair].append(ids)
    else:
        g = g1 if g1 != 0 else g2
        pts_tri = final_points[ids]
        fname = "unknown"
        for (a, val), name in zip(face_axis_value, face_names):
            if np.isclose(pts_tri[:, a], val, atol=1e-4).all():
                fname = name
                break
        p_name = f"cap_{g}_{fname}"
        cap_triangles.append(ids)
        cap_patch_names.append(p_name)
        cap_grain_tags.append(g)

for pair, tris in int_pair_to_tris.items():
    g_min, g_max = pair
    edge_to_tids = defaultdict(list)
    for tid, tri in enumerate(tris):
        for i in range(3):
            e = (min(tri[i], tri[(i+1)%3]), max(tri[i], tri[(i+1)%3]))
            edge_to_tids[e].append(tid)
    adj = defaultdict(set)
    for e, tids in edge_to_tids.items():
        if len(tids) == 2:
            adj[tids[0]].add(tids[1])
            adj[tids[1]].add(tids[0])
    visited = set()
    components = []
    for tid in range(len(tris)):
        if tid not in visited:
            comp = []
            queue = [tid]
            visited.add(tid)
            while queue:
                curr = queue.pop(0)
                comp.append(curr)
                for neigh in adj[curr]:
                    if neigh not in visited:
                        visited.add(neigh)
                        queue.append(neigh)
            components.append(comp)
            
    if len(components) == 1:
        p_name = f"int_({g_min},{g_max})"
        for tid in components[0]:
            interior_triangles.append(tris[tid])
            interior_patch_names.append(p_name)
            interior_grain_pairs.append(pair)
    else:
        for c_idx, comp in enumerate(components):
            p_name = f"int_({g_min},{g_max})_c{c_idx}"
            for tid in comp:
                interior_triangles.append(tris[tid])
                interior_patch_names.append(p_name)
                interior_grain_pairs.append(pair)

all_triangles = np.array(interior_triangles + cap_triangles, dtype=np.int64)
all_patch_names = interior_patch_names + cap_patch_names
unique_patch_names = sorted(list(set(all_patch_names)))
patch_name_to_id = {name: idx + 1 for idx, name in enumerate(unique_patch_names)}
triangle_patch_ids = np.array([patch_name_to_id[name] for name in all_patch_names], dtype=np.int32)

vertex_patch_map = defaultdict(set)
for tri, p_name in zip(all_triangles, all_patch_names):
    for v in tri:
        vertex_patch_map[v].add(p_name)

is_boundary_vertex = np.array([len(vertex_patch_map[v]) >= 2 for v in range(len(final_points))], dtype=bool)
print(f"=== Surface Patch Segmentation ===")
print(f"  Total surface points:     {len(final_points)}")
print(f"  Total surface triangles:  {len(all_triangles)}")
print(f"  Unique surface patches:   {len(unique_patch_names)}")
print(f"  Boundary (pinned) nodes:  {int(is_boundary_vertex.sum())} / {len(final_points)}")


## 11. Gmsh Surface Discrete-Entity Registration & Quality Diagnostics


In [ ]:
gmsh.initialize()
gmsh.option.setNumber('General.Terminal', 0)
gmsh.option.setNumber('General.Verbosity', 1)
gmsh.model.add("baseline_surface_patches")

pt_entity = gmsh.model.addDiscreteEntity(0)
node_tags = np.arange(1, len(final_points) + 1, dtype=np.int64)
gmsh.model.mesh.addNodes(0, pt_entity, node_tags.tolist(), final_points.ravel().tolist())

patch_to_tri_indices = defaultdict(list)
for idx, name in enumerate(all_patch_names):
    patch_to_tri_indices[name].append(idx)

all_el_tags = []
for p_name in unique_patch_names:
    surf_tag = gmsh.model.addDiscreteEntity(2)
    gmsh.model.setEntityName(2, surf_tag, str(p_name))
    p_tri_indices = patch_to_tri_indices[p_name]
    p_tris = all_triangles[p_tri_indices]
    el_tags = np.array(p_tri_indices, dtype=np.int64) + 1
    el_node_tags = (p_tris + 1).ravel().tolist()
    gmsh.model.mesh.addElementsByType(surf_tag, 2, el_tags.tolist(), el_node_tags)
    all_el_tags.extend(el_tags.tolist())

all_baseline_sicn = np.array(gmsh.model.mesh.getElementQualities(all_el_tags, "minSICN"))
valid_sicn = all_baseline_sicn[~np.isnan(all_baseline_sicn)]
print("=== Gmsh Discrete Surface Registration ===")
print(f"  Registered Patches:   {len(unique_patch_names)}")
print(f"  Total Triangles:      {len(all_triangles):,}")
print(f"  minSICN (Mean):       {np.mean(valid_sicn):.4f}")
print(f"  minSICN (Min):        {np.min(valid_sicn):.4f}")
gmsh.finalize()


## 12. Conformal Surface Patch Remeshing: Dynamic Edge Collapse & Split with 3D Lawson Optimization


In [ ]:
def fast_cross(v1, v2):
    return (v1[1]*v2[2] - v1[2]*v2[1], v1[2]*v2[0] - v1[0]*v2[2], v1[0]*v2[1] - v1[1]*v2[0])

def fast_norm(v):
    return math.sqrt(v[0]*v[0] + v[1]*v[1] + v[2]*v[2])

def fast_dot(v1, v2):
    return v1[0]*v2[0] + v1[1]*v2[1] + v1[2]*v2[2]

def fast_triangle_normal(p0, p1, p2):
    v1 = (p1[0] - p0[0], p1[1] - p0[1], p1[2] - p0[2])
    v2 = (p2[0] - p0[0], p2[1] - p0[1], p2[2] - p0[2])
    c = fast_cross(v1, v2)
    n = fast_norm(c)
    return (c[0]/n, c[1]/n, c[2]/n) if n > 1e-12 else (0.0, 0.0, 1.0)

def fast_min_angle_deg(p0, p1, p2):
    a2 = (p1[0]-p2[0])**2 + (p1[1]-p2[1])**2 + (p1[2]-p2[2])**2
    b2 = (p0[0]-p2[0])**2 + (p0[1]-p2[1])**2 + (p0[2]-p2[2])**2
    c2 = (p0[0]-p1[0])**2 + (p0[1]-p1[1])**2 + (p0[2]-p1[2])**2
    a, b, c = math.sqrt(a2), math.sqrt(b2), math.sqrt(c2)
    if a < 1e-12 or b < 1e-12 or c < 1e-12:
        return 0.0
    cos_A = max(-1.0, min(1.0, (b2 + c2 - a2) / (2.0 * b * c)))
    cos_B = max(-1.0, min(1.0, (a2 + c2 - b2) / (2.0 * a * c)))
    cos_C = max(-1.0, min(1.0, (a2 + b2 - c2) / (2.0 * a * b)))
    return min(math.acos(cos_A), math.acos(cos_B), math.acos(cos_C)) * 57.29577951308232

def collapse_interior_edge_patch(pts_dict, tris, boundary_edges_set, min_angle_target=30.0, max_collapses=15):
    b_verts = set(v for e in boundary_edges_set for v in e)
    edge_to_tris = defaultdict(list)
    for tid, t in enumerate(tris):
        if t is None: continue
        for i in range(3):
            edge_to_tris[(min(t[i], t[(i+1)%3]), max(t[i], t[(i+1)%3]))].append(tid)

    candidate_edges = [e for e, tids in edge_to_tris.items() if len(tids) == 2 and e not in boundary_edges_set]
    if not candidate_edges:
        return tris, 0
    candidate_edges.sort(key=lambda e: (pts_dict[e[0]][0]-pts_dict[e[1]][0])**2 + (pts_dict[e[0]][1]-pts_dict[e[1]][1])**2 + (pts_dict[e[0]][2]-pts_dict[e[1]][2])**2)
    collapses = 0
    for e in candidate_edges:
        if collapses >= max_collapses: break
        u, v = e
        if u in b_verts and v in b_verts: continue
        tids = [tid for tid in edge_to_tris.get(e, []) if tris[tid] is not None]
        if len(tids) != 2: continue
        t1_id, t2_id = tids[0], tids[1]
        t1, t2 = tris[t1_id], tris[t2_id]
        opp1 = [vid for vid in t1 if vid != u and vid != v]
        opp2 = [vid for vid in t2 if vid != u and vid != v]
        if len(opp1) != 1 or len(opp2) != 1: continue
        w1, w2 = opp1[0], opp2[0]
        u_tids = [tid for tid, t in enumerate(tris) if t is not None and u in t]
        v_tids = [tid for tid, t in enumerate(tris) if t is not None and v in t]
        u_neigh = set(vid for tid in u_tids for vid in tris[tid] if vid != u)
        v_neigh = set(vid for tid in v_tids for vid in tris[tid] if vid != v)
        if u_neigh.intersection(v_neigh) != {w1, w2}: continue

        if u in b_verts:
            target_id, remove_id, target_pos = u, v, pts_dict[u]
            mod_tids = [tid for tid in v_tids if tid != t1_id and tid != t2_id]
        elif v in b_verts:
            target_id, remove_id, target_pos = v, u, pts_dict[v]
            mod_tids = [tid for tid in u_tids if tid != t1_id and tid != t2_id]
        else:
            target_id, remove_id = u, v
            target_pos = tuple(0.5*(pts_dict[u][i] + pts_dict[v][i]) for i in range(3))
            mod_tids = [tid for tid in (u_tids + v_tids) if tid != t1_id and tid != t2_id]

        old_min_q = min(fast_min_angle_deg(pts_dict[tris[tid][0]], pts_dict[tris[tid][1]], pts_dict[tris[tid][2]]) for tid in (u_tids + v_tids))
        if old_min_q >= min_angle_target: continue
        temp_pts = dict(pts_dict)
        temp_pts[target_id] = target_pos
        new_angles = []
        valid = True
        for tid in mod_tids:
            new_t = [target_id if vid == remove_id else vid for vid in tris[tid]]
            if len(set(new_t)) < 3:
                valid = False; break
            n_old = fast_triangle_normal(pts_dict[tris[tid][0]], pts_dict[tris[tid][1]], pts_dict[tris[tid][2]])
            n_new = fast_triangle_normal(temp_pts[new_t[0]], temp_pts[new_t[1]], temp_pts[new_t[2]])
            if fast_dot(n_new, n_old) < 0.6:
                valid = False; break
            new_angles.append(fast_min_angle_deg(temp_pts[new_t[0]], temp_pts[new_t[1]], temp_pts[new_t[2]]))
        if not valid: continue
        new_min_q = min(new_angles) if len(new_angles) else 60.0
        if new_min_q > old_min_q + 0.5:
            pts_dict[target_id] = target_pos
            tris[t1_id] = None
            tris[t2_id] = None
            for tid in mod_tids:
                tris[tid] = [target_id if vid == remove_id else vid for vid in tris[tid]]
            collapses += 1
    return [t for t in tris if t is not None], collapses

def split_interior_obtuse_edge_patch(pts_dict, tris, boundary_edges_set, get_next_vid, max_splits=10):
    edge_to_tris = defaultdict(list)
    for tid, t in enumerate(tris):
        if t is None: continue
        for i in range(3):
            edge_to_tris[(min(t[i], t[(i+1)%3]), max(t[i], t[(i+1)%3]))].append(tid)
    splits = 0
    for tid in range(len(tris)):
        if splits >= max_splits: break
        t = tris[tid]
        if t is None: continue
        p0, p1, p2 = pts_dict[t[0]], pts_dict[t[1]], pts_dict[t[2]]
        a2 = (p1[0]-p2[0])**2 + (p1[1]-p2[1])**2 + (p1[2]-p2[2])**2
        b2 = (p0[0]-p2[0])**2 + (p0[1]-p2[1])**2 + (p0[2]-p2[2])**2
        c2 = (p0[0]-p1[0])**2 + (p0[1]-p1[1])**2 + (p0[2]-p1[2])**2
        a, b, c = math.sqrt(a2), math.sqrt(b2), math.sqrt(c2)
        if a < 1e-12 or b < 1e-12 or c < 1e-12: continue
        cos_A = max(-1.0, min(1.0, (b2 + c2 - a2) / (2.0 * b * c)))
        cos_B = max(-1.0, min(1.0, (a2 + c2 - b2) / (2.0 * a * c)))
        cos_C = max(-1.0, min(1.0, (a2 + b2 - c2) / (2.0 * a * b)))
        ang_A, ang_B, ang_C = math.acos(cos_A)*57.2957795, math.acos(cos_B)*57.2957795, math.acos(cos_C)*57.2957795
        if ang_A > 115.0: opp_edge = (min(t[1], t[2]), max(t[1], t[2]))
        elif ang_B > 115.0: opp_edge = (min(t[0], t[2]), max(t[0], t[2]))
        elif ang_C > 115.0: opp_edge = (min(t[0], t[1]), max(t[0], t[1]))
        else: continue
        if opp_edge in boundary_edges_set: continue
        tids = [i for i in edge_to_tris.get(opp_edge, []) if tris[i] is not None]
        if len(tids) != 2: continue
        t1_id, t2_id = tids[0], tids[1]
        u, v = opp_edge
        t1, t2 = tris[t1_id], tris[t2_id]
        w1_list = [x for x in t1 if x != u and x != v]
        w2_list = [x for x in t2 if x != u and x != v]
        if len(w1_list) != 1 or len(w2_list) != 1: continue
        w1, w2 = w1_list[0], w2_list[0]
        m_pos = tuple(0.5*(pts_dict[u][i] + pts_dict[v][i]) for i in range(3))
        m_id = get_next_vid()
        pts_dict[m_id] = m_pos
        tris[t1_id] = [u, m_id, w1]
        tris.append([m_id, v, w1])
        tris[t2_id] = [v, m_id, w2]
        tris.append([m_id, u, w2])
        splits += 1
    return [t for t in tris if t is not None], splits

def lawson_flip_3d_patch_queue(pts_dict, tris_list, boundary_edges_set, max_flips=200):
    tris = [list(t) for t in tris_list]
    edge_to_tris = defaultdict(list)
    for tid, t in enumerate(tris):
        for i in range(3):
            edge_to_tris[(min(t[i], t[(i+1)%3]), max(t[i], t[(i+1)%3]))].append(tid)
    queue = deque([e for e, tids in edge_to_tris.items() if len(tids) == 2 and e not in boundary_edges_set])
    in_queue = set(queue)
    flips_count = 0
    while queue and flips_count < max_flips:
        e = queue.popleft()
        in_queue.discard(e)
        tri_refs = edge_to_tris.get(e, [])
        if len(tri_refs) != 2 or e in boundary_edges_set: continue
        tid1, tid2 = tri_refs[0], tri_refs[1]
        t1, t2 = tris[tid1], tris[tid2]
        u, v = e
        if u not in t1 or v not in t1 or u not in t2 or v not in t2: continue
        w1_list = [x for x in t1 if x != u and x != v]
        w2_list = [x for x in t2 if x != u and x != v]
        if len(w1_list) != 1 or len(w2_list) != 1: continue
        w1, w2 = w1_list[0], w2_list[0]
        if w1 == w2: continue
        e_new = (min(w1, w2), max(w1, w2))
        if e_new in edge_to_tris: continue
        p_u, p_v, p_w1, p_w2 = pts_dict[u], pts_dict[v], pts_dict[w1], pts_dict[w2]
        min_curr = min(fast_min_angle_deg(p_u, p_v, p_w1), fast_min_angle_deg(p_v, p_u, p_w2))
        n1 = fast_triangle_normal(p_u, p_v, p_w1)
        n2 = fast_triangle_normal(p_v, p_u, p_w2)
        nA = fast_triangle_normal(p_u, p_w1, p_w2)
        nB = fast_triangle_normal(p_v, p_w2, p_w1)
        valid = (fast_dot(nA, n1) >= 0.7 and fast_dot(nB, n2) >= 0.7 and fast_dot(nA, nB) >= 0.5)
        if not valid:
            nA = fast_triangle_normal(p_u, p_w2, p_w1)
            nB = fast_triangle_normal(p_v, p_w1, p_w2)
            valid = (fast_dot(nA, n1) >= 0.7 and fast_dot(nB, n2) >= 0.7 and fast_dot(nA, nB) >= 0.5)
            if not valid: continue
        min_new = min(fast_min_angle_deg(p_u, p_w1, p_w2), fast_min_angle_deg(p_v, p_w2, p_w1))
        if min_new > min_curr + 0.5:
            del edge_to_tris[e]
            tris[tid1] = [u, w1, w2]
            tris[tid2] = [v, w2, w1]
            edge_to_tris[e_new] = [tid1, tid2]
            for old_e in [(min(v, w1), max(v, w1))]:
                if tid1 in edge_to_tris[old_e]: edge_to_tris[old_e].remove(tid1)
                if tid2 not in edge_to_tris[old_e]: edge_to_tris[old_e].append(tid2)
            for old_e in [(min(u, w2), max(u, w2))]:
                if tid2 in edge_to_tris[old_e]: edge_to_tris[old_e].remove(tid2)
                if tid1 not in edge_to_tris[old_e]: edge_to_tris[old_e].append(tid1)
            flips_count += 1
            for qe in [(min(u, w1), max(u, w1)), (min(w1, v), max(w1, v)), (min(v, w2), max(v, w2)), (min(w2, u), max(w2, u))]:
                if qe not in boundary_edges_set and len(edge_to_tris.get(qe, [])) == 2 and qe not in in_queue:
                    queue.append(qe); in_queue.add(qe)
    return tris, flips_count

def laplacian_relax_patch(pts_dict, tris, boundary_vertices_set, n_iter=4):
    adj = defaultdict(set)
    for t in tris:
        for i in range(3):
            adj[t[i]].add(t[(i+1)%3])
            adj[t[i]].add(t[(i+2)%3])
    free_nodes = [v for v in pts_dict if v not in boundary_vertices_set]
    if not free_nodes: return pts_dict
    curr = dict(pts_dict)
    for _ in range(n_iter):
        for v in free_nodes:
            nbrs = adj[v]
            if not nbrs: continue
            curr[v] = (0.6*curr[v][0] + 0.4*sum(curr[n][0] for n in nbrs)/len(nbrs),
                       0.6*curr[v][1] + 0.4*sum(curr[n][1] for n in nbrs)/len(nbrs),
                       0.6*curr[v][2] + 0.4*sum(curr[n][2] for n in nbrs)/len(nbrs))
    return curr

def seal_topological_micro_cracks(pts, tris, patch_names):
    edge_count = defaultdict(list)
    for tid, t in enumerate(tris):
        for i in range(3):
            edge_count[(min(t[i], t[(i+1)%3]), max(t[i], t[(i+1)%3]))].append(tid)
    open_edges = [e for e, tids in edge_count.items() if len(tids) == 1]
    if not open_edges: return tris, patch_names, 0
    adj = defaultdict(list)
    for u, v in open_edges:
        adj[u].append(v); adj[v].append(u)
    new_triangles = list(tris)
    new_patch_names = list(patch_names)
    sealed_count = 0
    visited = set()
    for start in adj:
        if start in visited: continue
        loop = [start]; visited.add(start)
        curr, prev = start, None
        while True:
            nxt = [n for n in adj[curr] if n != prev]
            if not nxt: break
            n_node = nxt[0]
            if n_node == start or n_node in visited: break
            visited.add(n_node); loop.append(n_node)
            prev, curr = curr, n_node
        if len(loop) == 3:
            new_triangles.append(loop)
            new_patch_names.append(patch_names[edge_count[(min(loop[0], loop[1]), max(loop[0], loop[1]))][0]])
            sealed_count += 1
        elif len(loop) == 4:
            p0, p1, p2, p3 = loop
            new_triangles.append([p0, p1, p2])
            new_triangles.append([p0, p2, p3])
            p_cand = patch_names[edge_count[(min(p0, p1), max(p0, p1))][0]]
            new_patch_names.extend([p_cand, p_cand])
            sealed_count += 2
    return np.array(new_triangles), np.array(new_patch_names), sealed_count

patch_to_tri_indices = defaultdict(list)
for idx, name in enumerate(all_patch_names):
    patch_to_tri_indices[name].append(idx)

global_points_list = [tuple(p) for p in final_points]
next_vid_counter = len(global_points_list)
def get_next_vid():
    global next_vid_counter
    vid = next_vid_counter
    next_vid_counter += 1
    return vid

patch_remesh_records = {}
patch_triangles_final = {}
n_accepted = 0
n_retained = 0
total_interior_flips = 0
total_interior_collapses = 0
total_interior_splits = 0

print("Optimizing conformal surface patches (Collapse + Split + 3D Lawson Queue)...")
for p_idx, p_name in enumerate(unique_patch_names):
    p_tri_indices = patch_to_tri_indices[p_name]
    p_tris_global = all_triangles[p_tri_indices]
    orig_angles = [fast_min_angle_deg(final_points[t[0]], final_points[t[1]], final_points[t[2]]) for t in p_tris_global]
    q_orig_min = float(min(orig_angles)) if len(orig_angles) else 0.0
    q_orig_mean = float(sum(orig_angles) / len(orig_angles)) if len(orig_angles) else 0.0

    if p_name.startswith("int_"):
        if len(p_tris_global) <= 2:
            patch_triangles_final[p_name] = p_tris_global.tolist()
            n_retained += 1
        else:
            edge_cnt = defaultdict(int)
            for t in p_tris_global:
                for i in range(3):
                    edge_cnt[(min(t[i], t[(i+1)%3]), max(t[i], t[(i+1)%3]))] += 1
            boundary_edges_patch = set(e for e, cnt in edge_cnt.items() if cnt != 2)
            boundary_verts_patch = set(v for e in boundary_edges_patch for v in e)
            has_internal_edges = any(cnt == 2 for cnt in edge_cnt.values())

            if not has_internal_edges or q_orig_min >= 35.0:
                patch_triangles_final[p_name] = p_tris_global.tolist()
                n_retained += 1
            else:
                pts_patch = {vid: global_points_list[vid] for vid in np.unique(p_tris_global)}
                current_tris = p_tris_global.tolist()
                current_tris, colls = collapse_interior_edge_patch(pts_patch, current_tris, boundary_edges_patch, min_angle_target=30.0, max_collapses=15)
                total_interior_collapses += colls
                current_tris, splits = split_interior_obtuse_edge_patch(pts_patch, current_tris, boundary_edges_patch, get_next_vid, max_splits=10)
                total_interior_splits += splits
                remeshed_tris, flips = lawson_flip_3d_patch_queue(pts_patch, current_tris, boundary_edges_patch, max_flips=200)
                total_interior_flips += flips
                if flips > 0 or splits > 0:
                    pts_patch = laplacian_relax_patch(pts_patch, remeshed_tris, boundary_verts_patch, n_iter=4)
                cand_angles = [fast_min_angle_deg(pts_patch[t[0]], pts_patch[t[1]], pts_patch[t[2]]) for t in remeshed_tris]
                q_cand_min = float(min(cand_angles)) if len(cand_angles) else 0.0
                q_cand_mean = float(sum(cand_angles) / len(cand_angles)) if len(cand_angles) else 0.0
                improved = (flips > 0 or colls > 0 or splits > 0) and (q_cand_min >= q_orig_min - 0.2 or q_cand_mean > q_orig_mean + 0.1)
                if improved:
                    for vid, pos in pts_patch.items():
                        if vid < len(global_points_list):
                            if vid not in boundary_verts_patch: global_points_list[vid] = pos
                        else:
                            while len(global_points_list) <= vid: global_points_list.append(pos)
                    patch_triangles_final[p_name] = remeshed_tris
                    n_accepted += 1
                else:
                    patch_triangles_final[p_name] = p_tris_global.tolist()
                    n_retained += 1
    else:
        patch_triangles_final[p_name] = p_tris_global.tolist()

optimized_points = np.array(global_points_list, dtype=np.float64)
optimized_triangles = []
optimized_patch_names = []
for p_name in unique_patch_names:
    for t in patch_triangles_final[p_name]:
        optimized_triangles.append(t)
        optimized_patch_names.append(p_name)
optimized_triangles = np.array(optimized_triangles, dtype=np.int64)
optimized_patch_names = np.array(optimized_patch_names)
optimized_triangles, optimized_patch_names, n_sealed = seal_topological_micro_cracks(optimized_points, optimized_triangles, optimized_patch_names)

print(f"=== Surface Patch Remeshing Results (Technique 2 Engine) ===")
print(f"  Total interior edge collapses: {total_interior_collapses}")
print(f"  Total interior edge splits:    {total_interior_splits}")
print(f"  Total 3D Lawson edge flips:    {total_interior_flips}")
print(f"  Patches optimized & accepted:  {n_accepted}")
print(f"  Patches retained/preserved:    {n_retained}")


## 13. Pre-Cursor to 3D Tet Mesh: Closed Grain Surface Loops & Volumes

Registers discrete surface entities and assembles closed `SurfaceLoop`s and 3D Volume entities per grain.


In [ ]:
t_start_c30 = time.perf_counter()

gmsh.initialize()
gmsh.option.setNumber('General.Terminal', 0)
gmsh.option.setNumber('General.Verbosity', 1)
gmsh.model.add("multigrain_conformal_complex")

# 1. Global node table (0D discrete entity)
pt_entity = gmsh.model.addDiscreteEntity(0)
node_tags = np.arange(1, len(optimized_points) + 1, dtype=np.int64)
gmsh.model.mesh.addNodes(0, pt_entity, node_tags.tolist(), optimized_points.ravel().tolist())

# 2. Group optimized triangles by patch name
patch_to_opt_tri_indices = defaultdict(list)
for idx, name in enumerate(optimized_patch_names):
    patch_to_opt_tri_indices[name].append(idx)

final_surface_tags = {}
grain_to_bounding_surfs = defaultdict(list)

for p_name in unique_patch_names:
    surf_tag = gmsh.model.addDiscreteEntity(2)
    gmsh.model.setEntityName(2, surf_tag, str(p_name))
    final_surface_tags[p_name] = surf_tag
    
    p_tri_indices = patch_to_opt_tri_indices[p_name]
    p_tris = optimized_triangles[p_tri_indices]
    
    el_tags = np.array(p_tri_indices, dtype=np.int64) + 1
    el_node_tags = (p_tris + 1).ravel().tolist()
    gmsh.model.mesh.addElementsByType(surf_tag, 2, el_tags.tolist(), el_node_tags)
    
    if p_name.startswith("int_"):
        clean_part = p_name[5:].split(")")[0]
        parts = clean_part.split(",")
        g_min, g_max = int(parts[0]), int(parts[1])
        grain_to_bounding_surfs[g_min].append(1 * surf_tag)
        grain_to_bounding_surfs[g_max].append(-1 * surf_tag)
    elif p_name.startswith("cap_"):
        parts = p_name.split("_")
        g = int(parts[1])
        grain_to_bounding_surfs[g].append(1 * surf_tag)

# 3. Assemble closed SurfaceLoops and 3D Volume entities per grain
grain_volumes = {}
for gid in grain_ids:
    bounding_surfs = grain_to_bounding_surfs.get(gid, [])
    if bounding_surfs:
        try:
            s_loop = gmsh.model.geo.addSurfaceLoop(bounding_surfs)
            vol = gmsh.model.geo.addVolume([s_loop])
            grain_volumes[gid] = vol
        except Exception as e:
            print(f"Warning: could not create volume for grain {gid}: {e}")

# Add physical groups for each grain (named 'grain_<gid>')
for gid, vol_tag in grain_volumes.items():
    pg = gmsh.model.addPhysicalGroup(3, [int(vol_tag)], tag=int(gid))
    gmsh.model.setPhysicalName(3, pg, f"grain_{gid}")

gmsh.model.geo.synchronize()
print(f"=== Gmsh 3D Multi-Grain Complex Assembly ===")
print(f"  Total Grain Volumes Created: {len(grain_volumes)} / {len(grain_ids)}")
print(f"  Total Discrete Surfaces:     {len(final_surface_tags)}")
print(f"  Global Nodes in Gmsh Model:  {len(optimized_points):,}")
print(f"  Section 13 Exec Time:        {time.perf_counter() - t_start_c30:.2f} s")


## 14. Boundary-to-Center Sizing Field Engine & 3D Conformal Tetrahedral Meshing

This section builds the active 3D tetrahedral sizing field according to `SIZING_MODE`:
- `constant_auto`: Automatically derived from surface triangle edge lengths.
- `linear`: Linear distance ramp via Gmsh `Threshold` field.
- `exponential`: Geometric growth factor per layer via `MathEval` field.
- `sigmoid`: $C^1$ Hermite smoothstep S-curve transition.
- `stepped_abs`: Stepped absolute distance field `{1: [x1, a1], 2: [x2, a2], 3: a3}`.
- `stepped_deq`: Grain-size-adaptive stepped zones normalized by grain equivalent diameter $D_{\text{eq}}$.
- Universal Upper Threshold: Global ceiling `H_MAX_ALLOWED` capping maximum tetrahedron edge length.
- Generates 3D tetrahedral mesh via `gmsh.model.mesh.generate(3)` with optional Netgen optimization.


In [ ]:
# ---------------------------------------------------------------------------
# Section 14: Sizing Field Engine & 3D Conformal Tetrahedral Meshing
# ---------------------------------------------------------------------------
t_start_3d = time.perf_counter()
print(f"=== Configuring 3D Sizing Field Engine (Mode: '{SIZING_MODE}') ===")

# 1. Calculate baseline surface triangle edge lengths
v0 = optimized_points[optimized_triangles[:, 0]]
v1 = optimized_points[optimized_triangles[:, 1]]
v2 = optimized_points[optimized_triangles[:, 2]]
edge_lens = np.concatenate([
    np.linalg.norm(v1 - v0, axis=1),
    np.linalg.norm(v2 - v1, axis=1),
    np.linalg.norm(v0 - v2, axis=1)
])
h_auto_mean = float(np.mean(edge_lens))
h_auto_median = float(np.median(edge_lens))
print(f"  Surface Mesh Edge Statistics: Mean = {h_auto_mean:.3f}, Median = {h_auto_median:.3f}")

# 2. Build Sizing Field based on SIZING_MODE
all_gb_surfaces = list(final_surface_tags.values())
active_field_tag = None

if SIZING_MODE == 'constant_auto':
    h_target = min(h_auto_mean, H_MAX_ALLOWED)
    print(f"  Mode 'constant_auto': Setting uniform target size = {h_target:.3f}")
    gmsh.option.setNumber("Mesh.MeshSizeMin", h_target)
    gmsh.option.setNumber("Mesh.MeshSizeMax", h_target)
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 1)

elif SIZING_MODE in ['linear', 'exponential', 'sigmoid', 'stepped_abs']:
    # Compute Euclidean distance to all discrete grain boundary surface patches
    d_field = gmsh.model.mesh.field.add("Distance")
    gmsh.model.mesh.field.setNumbers(d_field, "FacesList", all_gb_surfaces)
    
    if SIZING_MODE == 'linear':
        print(f"  Mode 'linear': Ramp h=[{H_MIN}, {H_MAX}] across dist=[{D_MIN}, {D_MAX}] (cap={H_MAX_ALLOWED})")
        t_field = gmsh.model.mesh.field.add("Threshold")
        gmsh.model.mesh.field.setNumber(t_field, "InField", d_field)
        gmsh.model.mesh.field.setNumber(t_field, "SizeMin", H_MIN)
        gmsh.model.mesh.field.setNumber(t_field, "SizeMax", min(H_MAX, H_MAX_ALLOWED))
        gmsh.model.mesh.field.setNumber(t_field, "DistMin", D_MIN)
        gmsh.model.mesh.field.setNumber(t_field, "DistMax", D_MAX)
        active_field_tag = t_field

    elif SIZING_MODE == 'exponential':
        print(f"  Mode 'exponential': h = min({H_MAX_ALLOWED}, {H_MIN} * {EXP_GROWTH_RATIO}^(d/{EXP_D0}))")
        expr = f"Min({H_MAX_ALLOWED}, {H_MIN} * ({EXP_GROWTH_RATIO}^(F{d_field} / {EXP_D0})))"
        m_field = gmsh.model.mesh.field.add("MathEval")
        gmsh.model.mesh.field.setString(m_field, "F", expr)
        active_field_tag = m_field

    elif SIZING_MODE == 'sigmoid':
        print(f"  Mode 'sigmoid': C1 smoothstep h=[{H_MIN}, {H_MAX}] across dist=[{D_MIN}, {D_MAX}] (cap={H_MAX_ALLOWED})")
        t_expr = f"Max(0.0, Min(1.0, (F{d_field} - {D_MIN}) / {max(1e-5, D_MAX - D_MIN)}))"
        h_cap = min(H_MAX, H_MAX_ALLOWED)
        expr = f"Min({H_MAX_ALLOWED}, {H_MIN} + ({h_cap - H_MIN}) * (3*({t_expr})^2 - 2*({t_expr})^3))"
        s_field = gmsh.model.mesh.field.add("MathEval")
        gmsh.model.mesh.field.setString(s_field, "F", expr)
        active_field_tag = s_field

    elif SIZING_MODE == 'stepped_abs':
        print(f"  Mode 'stepped_abs': Dictionary = {STEPPED_ABS_DICT} (cap={H_MAX_ALLOWED})")
        terms = []
        sorted_keys = sorted(STEPPED_ABS_DICT.keys())
        prev_x = None
        for k in sorted_keys:
            val = STEPPED_ABS_DICT[k]
            if isinstance(val, (list, tuple)):
                curr_x, curr_a = val[0], min(val[1], H_MAX_ALLOWED)
                if prev_x is None:
                    terms.append(f"(F{d_field} < {curr_x}) * {curr_a}")
                else:
                    terms.append(f"(F{d_field} >= {prev_x} && F{d_field} < {curr_x}) * {curr_a}")
                prev_x = curr_x
            else:
                final_a = min(float(val), H_MAX_ALLOWED)
                if prev_x is not None:
                    terms.append(f"(F{d_field} >= {prev_x}) * {final_a}")
                else:
                    terms.append(f"{final_a}")
        step_expr = " + ".join(terms)
        step_field = gmsh.model.mesh.field.add("MathEval")
        gmsh.model.mesh.field.setString(step_field, "F", step_expr)
        active_field_tag = step_field

    gmsh.model.mesh.field.setAsBackgroundMesh(active_field_tag)
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)

elif SIZING_MODE == 'stepped_deq':
    print(f"  Mode 'stepped_deq': Grain-adaptive fraction of Deq = {STEPPED_DEQ_DICT} (cap={H_MAX_ALLOWED})")
    # 1. Compute Deq per grain: Deq = 2 * (3 * V_grain / (4 * pi))^(1/3)
    unique_gids, counts = np.unique(clean_lfi, return_counts=True)
    deq_map = {}
    for gid_val, cnt in zip(unique_gids, counts):
        deq_map[int(gid_val)] = float(2.0 * ((3.0 * cnt) / (4.0 * np.pi)) ** (1.0 / 3.0))
    
    # 2. Group grains into 12 Deq bins to generate high-performance Restrict fields
    deq_values = np.array([deq_map[int(g)] for g in unique_gids])
    print(f"  Deq Distribution: Min = {deq_values.min():.2f}, Mean = {deq_values.mean():.2f}, Max = {deq_values.max():.2f} voxels")
    
    # Global Euclidean distance field to GB surfaces
    d_field = gmsh.model.mesh.field.add("Distance")
    gmsh.model.mesh.field.setNumbers(d_field, "FacesList", all_gb_surfaces)
    
    # Create quantized Deq bins
    n_bins = 12
    bin_edges = np.linspace(deq_values.min(), deq_values.max() + 1e-3, n_bins + 1)
    bin_field_tags = []
    
    sorted_keys = sorted(STEPPED_DEQ_DICT.keys())
    for b_idx in range(n_bins):
        b_lo, b_hi = bin_edges[b_idx], bin_edges[b_idx + 1]
        b_gids = [int(g) for g in unique_gids if b_lo <= deq_map[int(g)] < b_hi]
        if not b_gids:
            continue
        b_vols = [grain_volumes[g] for g in b_gids if g in grain_volumes]
        if not b_vols:
            continue
            
        b_deq_mid = 0.5 * (b_lo + b_hi)
        # Construct stepped expression for this Deq bin: x_i = f_i * Deq_mid
        terms = []
        prev_x = None
        for k in sorted_keys:
            val = STEPPED_DEQ_DICT[k]
            if isinstance(val, (list, tuple)):
                curr_x = float(val[0]) * b_deq_mid
                curr_a = min(float(val[1]), H_MAX_ALLOWED)
                if prev_x is None:
                    terms.append(f"(F{d_field} < {curr_x:.4f}) * {curr_a:.4f}")
                else:
                    terms.append(f"(F{d_field} >= {prev_x:.4f} && F{d_field} < {curr_x:.4f}) * {curr_a:.4f}")
                prev_x = curr_x
            else:
                final_a = min(float(val), H_MAX_ALLOWED)
                if prev_x is not None:
                    terms.append(f"(F{d_field} >= {prev_x:.4f}) * {final_a:.4f}")
                else:
                    terms.append(f"{final_a:.4f}")
        b_expr = " + ".join(terms)
        
        m_f = gmsh.model.mesh.field.add("MathEval")
        gmsh.model.mesh.field.setString(m_f, "F", b_expr)
        
        r_f = gmsh.model.mesh.field.add("Restrict")
        gmsh.model.mesh.field.setNumber(r_f, "InField", m_f)
        gmsh.model.mesh.field.setNumbers(r_f, "VolumesList", b_vols)
        bin_field_tags.append(r_f)
    
    # Combine all volume-restricted sizing fields with Min field
    min_f = gmsh.model.mesh.field.add("Min")
    gmsh.model.mesh.field.setNumbers(min_f, "FieldsList", bin_field_tags)
    active_field_tag = min_f
    gmsh.model.mesh.field.setAsBackgroundMesh(active_field_tag)
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)

# 3. Configure 3D Algorithm & Universal Ceiling
gmsh.option.setNumber("Mesh.Algorithm3D", GMSH_3D_ALGORITHM) # 1: Delaunay, 10: HXT
gmsh.option.setNumber("Mesh.MeshSizeMax", H_MAX_ALLOWED)

# 4. Generate 3D Tetrahedral Conformal Mesh
print(f"\nGenerating 3D Conformal Tetrahedral Mesh (gmsh.model.mesh.generate(3), Algorithm3D={GMSH_3D_ALGORITHM})...")
t_gen_start = time.perf_counter()
gmsh.model.mesh.generate(3)
t_gen_elapsed = time.perf_counter() - t_gen_start
print(f"  3D Tet Generation Completed in {t_gen_elapsed:.2f} s")

# 5. Post-Meshing Netgen 3D Quality Optimization
if GMSH_OPTIMIZE_NETGEN:
    print("Optimizing 3D Tetrahedral Quality (gmsh.model.mesh.optimize('Netgen'))...")
    t_opt_start = time.perf_counter()
    try:
        gmsh.model.mesh.optimize("Netgen")
        print(f"  Netgen Optimization Completed in {time.perf_counter() - t_opt_start:.2f} s")
    except Exception as e:
        print(f"  Netgen optimization note: {e}")

# 6. Element Order Conversion (C3D4 -> C3D10)
if ELEMENT_ORDER == 2:
    print("Converting linear C3D4 tetrahedra to quadratic C3D10 (10-node)...")
    gmsh.model.mesh.setOrder(2)

# Extract 3D mesh element counts
n_total_nodes = len(gmsh.model.mesh.getNodes()[0])
n_tets = 0
for etyp, etags, _ in zip(*gmsh.model.mesh.getElements(dim=3)):
    if int(etyp) in [4, 11]: # 4: C3D4, 11: C3D10
        n_tets += len(etags)

print(f"\n=== 3D Conformal Mesh Assembly Summary ===")
print(f"  Total 3D Nodes:       {n_total_nodes:,}")
print(f"  Total 3D Tetrahedra:  {n_tets:,}")
print(f"  Element Type:         {'C3D4 (4-node linear)' if ELEMENT_ORDER == 1 else 'C3D10 (10-node quadratic)'}")
print(f"  Total Execution Time: {time.perf_counter() - t_start_3d:.2f} s")


## 15. Visualization of Remeshed Surface Patches


In [ ]:
faces_opt = np.hstack([np.full((len(all_triangles), 1), 3), all_triangles]).ravel()
remeshed_surf_pv = pv.PolyData(optimized_points, faces_opt)
remeshed_surf_pv.cell_data['patch_id'] = triangle_patch_ids

cell_grain_labels = np.zeros(len(all_triangles), dtype=np.int32)
for idx, p_name in enumerate(all_patch_names):
    if p_name.startswith("int_"):
        parts = p_name[5:-1].split(",")
        cell_grain_labels[idx] = int(parts[0])
    elif p_name.startswith("cap_"):
        parts = p_name.split("_")
        cell_grain_labels[idx] = int(parts[1])
remeshed_surf_pv.cell_data['grain'] = cell_grain_labels

pl = pv.Plotter(window_size=(900, 750))
pl.set_background('white')
pl.add_mesh(remeshed_surf_pv, scalars='patch_id', cmap='turbo', show_edges=False)
pl.add_title(f"Remeshed Conformal Multi-Grain Surface ({len(unique_patch_names)} patches)", font_size=11)
pl.add_axes()
pl.show()


## 16. Interactive Crinkle Clip Plane Visualization with Grain Boundary Junction Lines & Triple Points


In [ ]:
SHOW_JUNCTION_LINES = False
JUNCTION_LINE_WIDTH = 2
JUNCTION_LINE_COLOR = 'black'
INCLUDE_RVE_SEAMS   = True
SHOW_TRIPLE_POINTS  = False
TRIPLE_POINT_SIZE   = 5.0
TRIPLE_POINT_COLOR  = 'red'
TRIPLE_POINT_MODE   = 'nodes'

patch_grain_map = {}
for p_name in unique_patch_names:
    if p_name.startswith("int_"):
        clean_part = p_name[5:].split(")")[0]
        g1, g2 = map(int, clean_part.split(","))
        patch_grain_map[p_name] = (g1, g2)
    elif p_name.startswith("cap_"):
        g = int(p_name.split("_")[1])
        patch_grain_map[p_name] = (g, 0)
    else:
        patch_grain_map[p_name] = ()

edge_to_grains = defaultdict(set)
for tri, p_name in zip(all_triangles, all_patch_names):
    gr = patch_grain_map[p_name]
    for i in range(3):
        u, v = tri[i], tri[(i + 1) % 3]
        edge_to_grains[(min(u, v), max(u, v))].update(gr)

junction_edges = [e for e, g in edge_to_grains.items() if len(g if INCLUDE_RVE_SEAMS else [gid for gid in g if gid != 0]) >= 3]
n_j_edges = len(junction_edges)
junction_lines_pv = pv.PolyData()
if n_j_edges > 0:
    lines_conn = np.empty((n_j_edges, 3), dtype=np.int64)
    lines_conn[:, 0] = 2
    lines_conn[:, 1:] = junction_edges
    junction_lines_pv.points = optimized_points
    junction_lines_pv.lines = lines_conn.ravel()

j_deg = defaultdict(int)
for u, v in junction_edges:
    j_deg[u] += 1; j_deg[v] += 1
triple_point_vidxs = np.array([v for v, deg in j_deg.items() if deg >= 3], dtype=np.int64) if TRIPLE_POINT_MODE == 'nodes' else np.unique(junction_edges)
triple_points_pv = pv.PolyData(optimized_points[triple_point_vidxs]) if len(triple_point_vidxs) > 0 else pv.PolyData()

p_clip = pv.Plotter(window_size=(900, 750))
p_clip.set_background('white')
p_clip.add_mesh_clip_plane(remeshed_surf_pv, normal='x', crinkle=True, scalars='grain', cmap='tab20c', show_edges=True, edge_color='black', line_width=0.5)
if SHOW_JUNCTION_LINES and n_j_edges > 0:
    p_clip.add_mesh(junction_lines_pv, color=JUNCTION_LINE_COLOR, line_width=JUNCTION_LINE_WIDTH)
if SHOW_TRIPLE_POINTS and len(triple_point_vidxs) > 0:
    p_clip.add_mesh(triple_points_pv, color=TRIPLE_POINT_COLOR, point_size=TRIPLE_POINT_SIZE, render_points_as_spheres=True)
p_clip.add_title(f"Crinkle Clip Plane Sectioning -- ({len(grain_ids)} grains)", font_size=10)
p_clip.add_axes()
p_clip.show()


## 17. Watertightness Status & Topological Manifold Diagnostics


In [ ]:
print("==========================================================================")
print("     WATERTIGHTNESS & TOPOLOGICAL MANIFOLD DIAGNOSTICS")
print("==========================================================================")
fe_global = vtk.vtkFeatureEdges()
fe_global.SetInputData(stitched)
fe_global.BoundaryEdgesOn()
fe_global.FeatureEdgesOff()
fe_global.NonManifoldEdgesOff()
fe_global.ManifoldEdgesOff()
fe_global.Update()
n_open_global = fe_global.GetOutput().GetNumberOfLines()

print(f"1. OVERALL MULTI-GRAIN CONFORMAL COMPLEX:")
print(f"   Total Vertices          : {len(optimized_points):,}")
print(f"   Total Triangle Facets   : {len(optimized_triangles):,}")
print(f"   Total Surface Patches   : {len(unique_patch_names)}")
print(f"   Open Boundary Edges     : {n_open_global}")
print(f"   Watertight Status       : {'WATERTIGHT / CLOSED' if n_open_global == 0 else 'NOT WATERTIGHT'}")
assert n_open_global == 0, f"Fatal: {n_open_global} open edges detected on multi-grain complex!"


## 18. 3D Tetrahedral Mesh Quality Diagnostics

Comprehensive quality evaluation of the generated 3D tetrahedral elements:
1. **Tetrahedral Volume Distribution**: Strict verification that 100% of tetrahedra have positive volume ($V > 0$, zero inverted elements).
2. **Signed Inverse Condition Number (`minSICN`)**: Measures tetrahedral shape regularity (ideal equilateral = 1.0; $> 0.2$ is good; $> 0.4$ is excellent).
3. **Tetrahedral Aspect Ratio (Radius Ratio $R_{\text{in}} / R_{\text{circ}}$)**: Evaluates circumradius to inradius ratio.
4. **Element Sizing Distribution**: Validates that element sizes at grain boundaries match $h_{\text{min}}$ and expand smoothly towards $H_{\text{MAX}}$ in the grain cores.


In [ ]:
# ---------------------------------------------------------------------------
# Section 18: 3D Tetrahedral Mesh Quality Diagnostics
# ---------------------------------------------------------------------------
print("==========================================================================")
print("          3D TETRAHEDRAL MESH QUALITY & SIZING DIAGNOSTICS")
print("==========================================================================")

# 1. Extract all 3D mesh elements from Gmsh
node_tags, node_coords, _ = gmsh.model.mesh.getNodes()
nodes_3d = np.array(node_coords, dtype=np.float64).reshape(-1, 3)
node_tag_to_idx = {int(tag): idx for idx, tag in enumerate(node_tags)}

elem_types, elem_tags_list, elem_node_tags_list = gmsh.model.mesh.getElements(dim=3)
all_tet_tags = []
all_tet_conn = []

for etyp, etags, enodes in zip(elem_types, elem_tags_list, elem_node_tags_list):
    if int(etyp) == 4: # 4-node linear tetrahedron (C3D4)
        tags_arr = np.array(enodes).reshape(-1, 4)
        for t_tag, t_nodes in zip(etags, tags_arr):
            all_tet_tags.append(int(t_tag))
            all_tet_conn.append([node_tag_to_idx[int(n)] for n in t_nodes])
    elif int(etyp) == 11: # 10-node quadratic tetrahedron (C3D10)
        tags_arr = np.array(enodes).reshape(-1, 10)
        for t_tag, t_nodes in zip(etags, tags_arr):
            all_tet_tags.append(int(t_tag))
            # Use vertex nodes 0..3 for volume and geometry checks
            all_tet_conn.append([node_tag_to_idx[int(n)] for n in t_nodes[:4]])

all_tet_tags = np.array(all_tet_tags, dtype=np.int64)
all_tet_conn = np.array(all_tet_conn, dtype=np.int64)
n_tetrahedra = len(all_tet_conn)

# 2. Calculate Exact Tetrahedral Volumes: V = (1/6) * |det(v1-v0, v2-v0, v3-v0)|
p0 = nodes_3d[all_tet_conn[:, 0]]
p1 = nodes_3d[all_tet_conn[:, 1]]
p2 = nodes_3d[all_tet_conn[:, 2]]
p3 = nodes_3d[all_tet_conn[:, 3]]

d1 = p1 - p0
d2 = p2 - p0
d3 = p3 - p0

cross_d1_d2 = np.cross(d1, d2)
signed_vols = np.sum(cross_d1_d2 * d3, axis=1) / 6.0
tet_vols = np.abs(signed_vols)
n_inverted = int(np.sum(signed_vols <= 0.0))

# Equivalent linear element size h_tet = (6 * sqrt(2) * V)^(1/3)
h_tet_equiv = (8.48528137424 * tet_vols) ** (1.0 / 3.0)

# 3. Query Gmsh Signed Inverse Condition Number (SICN)
print("Querying element qualities (minSICN) from Gmsh...")
all_sicn = np.array(gmsh.model.mesh.getElementQualities(all_tet_tags.tolist(), "minSICN"))
valid_sicn = all_sicn[~np.isnan(all_sicn)]

# 4. Tabulate 3D Tetrahedral Quality Statistics
print(f"\n1. TETRAHEDRAL VOLUME & TOPOLOGY:")
print(f"   Total 3D Tetrahedra    : {n_tetrahedra:,}")
print(f"   Total 3D Mesh Nodes    : {len(nodes_3d):,}")
print(f"   Inverted Elements (<=0): {n_inverted} (target: 0)")
print(f"   Status                 : {'100% STRICTLY POSITIVE VOLUMES' if n_inverted == 0 else 'WARNING: Inverted elements detected'}")
print(f"   Min Element Volume     : {tet_vols.min():.4e} voxels^3")
print(f"   Mean Element Volume    : {tet_vols.mean():.4f} voxels^3")
print(f"   Median Element Volume  : {np.median(tet_vols):.4f} voxels^3")
print(f"   Max Element Volume     : {tet_vols.max():.4f} voxels^3")

print(f"\n2. ELEMENT SIZE GRADIENT (h_tet = V^(1/3)):")
print(f"   Min Element Size (h)   : {h_tet_equiv.min():.3f} voxels")
print(f"   Mean Element Size (h)  : {h_tet_equiv.mean():.3f} voxels")
print(f"   Median Element Size (h): {np.median(h_tet_equiv):.3f} voxels")
print(f"   Max Element Size (h)   : {h_tet_equiv.max():.3f} voxels (Capped by H_MAX_ALLOWED = {H_MAX_ALLOWED})")

print(f"\n3. SHAPE REGULARITY (minSICN Quality Metric):")
print(f"   Mean minSICN           : {np.mean(valid_sicn):.4f}")
print(f"   Median minSICN         : {np.median(valid_sicn):.4f}")
print(f"   5th Percentile minSICN : {np.percentile(valid_sicn, 5):.4f}")
print(f"   Absolute Min minSICN   : {np.min(valid_sicn):.4f}")
print(f"   Elements with SICN < 0.2: {int(np.sum(valid_sicn < 0.2)):,} ({np.mean(valid_sicn < 0.2)*100:.2f}%)")
print(f"   Elements with SICN >= 0.5: {int(np.sum(valid_sicn >= 0.5)):,} ({np.mean(valid_sicn >= 0.5)*100:.2f}%)")
print("==========================================================================")


## 19. Interactive 3D Tetrahedral Mesh Visualization (Grain IDs & Sizing Gradient)

Interactive sectioning tools displaying the volumetric 3D tetrahedral mesh:
1. **Grain ID View**: Demonstrates that the 3D tetrahedral mesh is strictly conformal across all internal grain boundaries.
2. **Sizing Gradient View**: Colors each tetrahedron by its local element volume / size $h$, visually validating the fine resolution at grain boundaries and smooth transition to coarser elements in the grain cores.


In [ ]:
# ---------------------------------------------------------------------------
# Section 19: Interactive 3D Tetrahedral Mesh Visualization
# ---------------------------------------------------------------------------
# 1. Map each tetrahedron to its parent Grain ID via Gmsh physical groups
tet_grain_ids = np.zeros(n_tetrahedra, dtype=np.int32)
tag_to_tet_idx = {tag: idx for idx, tag in enumerate(all_tet_tags)}

for gid in grain_ids:
    try:
        g_vols = gmsh.model.getEntitiesForPhysicalGroup(3, int(gid))
        for v_tag in g_vols:
            _, v_elem_tags, _ = gmsh.model.mesh.getElements(3, v_tag)
            if len(v_elem_tags) > 0:
                for el_t in v_elem_tags[0]:
                    if int(el_t) in tag_to_tet_idx:
                        tet_grain_ids[tag_to_tet_idx[int(el_t)]] = int(gid)
    except Exception:
        pass

# 2. Build PyVista UnstructuredGrid for the 3D volume mesh
cells_pv = np.hstack([np.full((n_tetrahedra, 1), 4), all_tet_conn]).ravel()
cell_types_pv = np.full(n_tetrahedra, pv.CellType.TETRA, dtype=np.uint8)
tet_mesh_pv = pv.UnstructuredGrid(cells_pv, cell_types_pv, nodes_3d)

# Attach cell scalar fields
tet_mesh_pv.cell_data['grain'] = tet_grain_ids
tet_mesh_pv.cell_data['tet_size'] = h_tet_equiv
tet_mesh_pv.cell_data['tet_volume'] = tet_vols
tet_mesh_pv.cell_data['sicn_quality'] = all_sicn

# ---------------------------------------------------------------------------
# Plot A: Crinkle-Clipped 3D Tet Mesh Colored by Grain ID
# ---------------------------------------------------------------------------
p_tet_grain = pv.Plotter(window_size=(900, 750))
p_tet_grain.set_background('white')
p_tet_grain.add_mesh_clip_plane(
    tet_mesh_pv,
    normal='x',
    crinkle=True,
    scalars='grain',
    cmap='tab20',
    show_edges=True,
    edge_color='black',
    line_width=0.4,
    scalar_bar_args={'title': 'Grain ID'}
)
p_tet_grain.add_title(f"3D Conformal Tet Mesh -- Grain ID Partition ({n_tetrahedra:,} tets, {len(grain_ids)} grains)", font_size=10)
p_tet_grain.add_axes()
p_tet_grain.show()

# ---------------------------------------------------------------------------
# Plot B: Crinkle-Clipped 3D Tet Mesh Colored by Element Sizing Gradient
# ---------------------------------------------------------------------------
p_tet_size = pv.Plotter(window_size=(900, 750))
p_tet_size.set_background('white')
p_tet_size.add_mesh_clip_plane(
    tet_mesh_pv,
    normal='x',
    crinkle=True,
    scalars='tet_size',
    cmap='coolwarm',
    show_edges=True,
    edge_color='black',
    line_width=0.4,
    scalar_bar_args={'title': 'Element Size h (voxels)'}
)
p_tet_size.add_title(f"3D Boundary-to-Center Sizing Gradient (Mode: '{SIZING_MODE}', Cap: {H_MAX_ALLOWED})", font_size=10)
p_tet_size.add_axes()
p_tet_size.show()

# Finalize Gmsh kernel cleanly
gmsh.finalize()
print("Gmsh kernel finalized cleanly. 3D Conformal Tetrahedral Meshing complete!")
